# PHI/PII De-identification of Scanned Documents — v6 (Textract + GPT-5.4 context + Azure DeID Surrogate)

Replaces PHI/PII **on the image** with fake values written by **Azure DeID's `Surrogate` operation**, keeping the original font, size, ink colour and layout. GPT-5.4 gives Azure the form context it needs; no local fake-value rules.

**Flow**

| Step | Section | What happens |
| --- | --- | --- |
| 1 | 4–6 | **Textract `DetectDocumentText`** reads the page: every word with its text, pixel box, tilt, confidence and printed/handwriting type → `full_text` + `offset_map` |
| 2 | 9b | **GPT-5.4** reads the page image: label → value pairs and PHI values, matched to the OCR words → `label: value` text (`ctx_text`) + GPT PHI hits |
| 3 | 10 | **Azure DeID `Surrogate`** on the page text and on the `label: value` text; every word Azure changed gets Azure's value (same value → same fake across the batch). PHI GPT found but Azure changed in neither text is erased |
| 4 | 11 | Each changed word is **erased and redrawn** with its new value in the original font, size, position and colour. Longer fakes use free space, push unchanged words along, then are condensed, then drawn smaller |
| 4b | 11b | **Side-by-side check** of the test image (original \| de-identified) before running the batch |
| 5 | 12–13 | **Batch**: every image in a folder → de-identified PNG, side-by-side comparison, text files, verification record, `batch_summary.csv` |

**Cost per page:** one Textract call (+ one for the verification record), one GPT-5.4 call with the image, two Azure DeID calls (`deid_input_bytes` in the summary).

**What is changed:** every word Azure changes in either text, and (erased, not faked) PHI GPT found that Azure did not change. Review the side-by-side images.

## 1. Import Required Libraries

Libraries used in several sections. Each later section imports what only it needs (`re`, `numpy`, `PIL`, `difflib`…).

| Library | Used for |
| --- | --- |
| `boto3` + `aws_client` (company wrapper) | AWS Textract OCR; `aws_client` signs in and renews expired credentials |
| `json`, `os` | reading API answers and settings |
| `Counter` | counting block types / actions for the printouts |
| `ssl`, `sys`, `tempfile` | building the company (Netskope) certificate file, section 7 |
| `urllib.request`, `urllib.error` | calling the Azure DeID REST API |

In [ ]:
import boto3        # AWS SDK for Python — Textract OCR
import json          # JSON parsing for API responses
import os            # File path operations
from collections import Counter  # Block type counting
from icp.authenticator.aws.boto3_services import boto3_clients as aws_client
import ssl           # Certificate handling
import sys           # Platform detection
import tempfile      # Temporary file creation
import urllib.error  # HTTP error handling
import urllib.request  # HTTP requests

print("✓ All libraries imported successfully")

## 1b. Debug Printouts

Printers for each stage. With `DEBUG = True` the single-image run prints:

| Stage | Section | Shows |
| --- | --- | --- |
| 1 | 6 | every OCR line, lowest word confidence, low-confidence and handwritten words |
| 1b | 9b | every field GPT-5.4 returned (label, value, category, is_phi) and whether it was found among the OCR words; free-text PHI |
| 2t | 10 | the texts in full: the page text sent to Azure and Azure's output, the GPT `label: value` text sent and Azure's output, and the final text on the output image |
| 2 / 2c | 10 | original → Azure's value per changed block and which text it came from; GPT PHI Azure did not change |
| 3 | 10 | every replaced value: original → fake |
| 4 | 11 | what happened to each word on the image (replaced, condensed, smaller font, moved, handwriting font) |

`DEBUG_BATCH = True` prints the same for every page of the batch (long output). `DEBUG_MAX_ROWS` caps each table.

**The printouts show real PHI.** Use them on your own machine only and clear the outputs before sharing the notebook.

In [ ]:
# Debug printouts for each stage. They contain real PHI: clear the outputs before sharing the notebook.
DEBUG = True            # single-image run (sections 6, 10, 11)
DEBUG_BATCH = False     # every page of the batch run (very long output)
DEBUG_MAX_ROWS = 120    # rows printed per table

def _dbg_title(t):
    print("\n" + "=" * 100 + f"\n{t}\n" + "=" * 100)

def _dbg_rows(rows, header):
    widths = [max(len(str(h)), *(len(str(r[i])) for r in rows)) if rows else len(str(h)) for i, h in enumerate(header)]
    widths = [min(w, 60) for w in widths]
    fmt = "  ".join("{:<%d}" % w for w in widths)
    print(fmt.format(*header)); print(fmt.format(*["-" * w for w in widths]))
    for r in rows[:DEBUG_MAX_ROWS]:
        print(fmt.format(*[str(x)[:60] for x in r]))
    if len(rows) > DEBUG_MAX_ROWS:
        print(f"... {len(rows) - DEBUG_MAX_ROWS} more rows")

def debug_ocr(offset_map):
    """Stage 1: what Textract read."""
    _dbg_title("STAGE 1 - OCR (Textract DetectDocumentText): lines as read")
    by_line = {}
    for w in offset_map:
        by_line.setdefault(w['line_idx'], []).append(w)
    rows = []
    for li, ws in sorted(by_line.items()):
        low = [w['text'] for w in ws if w['confidence'] < 70]
        hw = [w['text'] for w in ws if w['handwriting']]
        rows.append((li, f"{min(w['confidence'] for w in ws):.0f}", " ".join(w['text'] for w in ws),
                     ("LOW:" + ",".join(low)) if low else "", ("HAND:" + ",".join(hw)) if hw else ""))
    _dbg_rows(rows, ("line", "min_conf", "text", "low_conf_words", "handwriting"))
    print(f"\nwords: {len(offset_map)} | lines: {len(by_line)} | low-confidence (<70) words: "
          f"{sum(w['confidence'] < 70 for w in offset_map)} | handwritten: {sum(w['handwriting'] for w in offset_map)}")

def debug_surrogates(entities):
    """Stage 3: original -> fake for every replaced value."""
    _dbg_title("STAGE 3 - Replaced values: original -> fake")
    _dbg_rows([(e["kind"], e["original"], e.get("surrogate", ""), ",".join(e["sources"])) for e in entities],
              ("kind", "original", "fake", "found_by"))

def debug_render(actions, word_by_id):
    """Stage 4: what happened on the image."""
    _dbg_title("STAGE 4 - Image: action per changed word")
    _dbg_rows([(word_by_id[wid]['text'], word_by_id[wid].get('new_text', ''), a) for wid, a in actions.items()
               if wid in word_by_id], ("original", "drawn", "action"))

print("✓ Debug printers ready (DEBUG =", DEBUG, "| DEBUG_BATCH =", DEBUG_BATCH, ")")

## 2. Configuration (.env)

Reads the Azure DeID settings from `.env`, so no secret is written in the notebook.

| `.env` key | Required | Meaning |
| --- | --- | --- |
| `AZURE_DEID_ENDPOINT` | yes | your DeID service URL |
| `AZURE_DEID_TENANT_ID` | yes | Azure tenant for the sign-in |
| `AZURE_DEID_SCOPE` | no | token scope (default `https://deid.azure.com/.default`) |
| `AZURE_DEID_API_VERSION` | no | REST API version (default `2024-11-15`) |

A missing required key stops the cell at once with a `KeyError`.

The GPT-5.4 settings are read in section 9b from the same `.env`: `endpoint`, `deployment`, `subscription_key`, `api_version`, optional `model_name`.

In [ ]:
# Azure DeID Service Configuration — loaded from .env (keeps secrets out of the notebook)
from dotenv import load_dotenv

load_dotenv()  # reads the .env file in the project root

ENDPOINT = os.environ["AZURE_DEID_ENDPOINT"]
TENANT_ID = os.environ["AZURE_DEID_TENANT_ID"]
SCOPE = os.environ.get("AZURE_DEID_SCOPE", "https://deid.azure.com/.default")
API_VERSION = os.environ.get("AZURE_DEID_API_VERSION", "2024-11-15")

print(f"Endpoint: {ENDPOINT}")
print(f"Tenant ID: {TENANT_ID}")
print(f"API Version: {API_VERSION}")

## 3. Image File Handler and Loader

`load_image()` checks that the file exists and is a supported single-page image (`.png`, `.jpg`, `.jpeg`, `.bmp`, `.gif`). Multi-page PDFs / TIFFs must be split into page images first. `IMAGE_PATH` is the image used for the single-image test run (sections 4–11); the batch (section 12–13) finds its images itself.

In [ ]:
# Image file handler: single-page images only (PNG, JPG, BMP, GIF).
# Multi-page PDF / TIFF files must be split into page images first.

SUPPORTED_IMAGE_FORMATS = ['.png', '.jpg', '.jpeg', '.bmp', '.gif']

def load_image(image_path):
    """
    Load and validate an image file.
    
    Args:
        image_path: Full path to the image file
        
    Returns:
        image_path: Validated path to the image
        
    Raises:
        FileNotFoundError: If file doesn't exist
        ValueError: If file format is not supported
    """
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image file not found: {image_path}")
    
    file_ext = os.path.splitext(image_path)[1].lower()
    
    if file_ext not in SUPPORTED_IMAGE_FORMATS:
        raise ValueError(f"Unsupported format: {file_ext}. Supported formats: {SUPPORTED_IMAGE_FORMATS}")
    
    print(f"✓ Image loaded: {image_path}")
    return image_path


# Set your image path here
IMAGE_PATH = r"C:\Users\RSahu\Documents\Azure_deidentification\Input_docs\data\test\CMS 1500 202604230008171_page_1.png"
# Validate the image
IMAGE_PATH = load_image(IMAGE_PATH)

## 4. AWS Textract OCR (basic text detection)

`get_ocr()` sends the image bytes to Textract **`DetectDocumentText`** (about $0.0015 per page). It returns a flat list of blocks: one `PAGE`, one `LINE` per printed line, one `WORD` per word. Each block has its text, confidence, position (fractions of the page) and `TextType` (`PRINTED` / `HANDWRITING`); a LINE lists its WORD ids. No key-value pairs or tables are needed in v6.

- The image must be under 10 MB (Textract's limit for a direct call).
- `@aws_client.refresh_on_expire` renews expired AWS credentials and retries, which matters in long batches.
- The same `get_ocr` re-reads the output image for the verification record (section 12).

In [ ]:
with open(IMAGE_PATH, "rb") as f:
    image_bytes = f.read()

print(f"Image size: {len(image_bytes)/1024:.1f} KB")
assert len(image_bytes) < 10 * 1024 * 1024, "Image exceeds 10MB sync limit"
# Textract accepts at most 10 MB in a direct call
@aws_client.refresh_on_expire
def get_ocr(image_bytes):
    """Basic OCR: lines + words with boxes, confidence and handwriting flag (Textract DetectDocumentText)."""
    textract = aws_client.get("textract")
    return textract.detect_document_text(Document={'Bytes': image_bytes})

response = get_ocr(image_bytes)
blocks = response['Blocks']
print(f"Total blocks returned: {len(blocks)}")

type_counts = Counter(b['BlockType'] for b in blocks)
print("Block type counts:", dict(type_counts))

## 5. LINE Blocks (check only)

Prints every LINE with its confidence and position, and saves the same list to `extracted_text_detailed.txt`, so you can see what Textract read. Nothing later uses this file or these cells.

In [ ]:
# For checking only: every LINE with confidence and position (printed and saved to a text file)

lines = [b for b in blocks if b['BlockType'] == 'LINE']
print(f"Total lines detected: {len(lines)}\n")

output_lines = []

for line in lines:
    text = line['Text']
    conf = line['Confidence']
    block_type = line['BlockType']
    bbox = line['Geometry']['BoundingBox']  # normalized 0-1 coords: Left, Top, Width, Height

    entry = (
        f"Type: {block_type} | Confidence: {conf:.2f}% | "
        f"Location(Left={bbox['Left']:.4f}, Top={bbox['Top']:.4f}, "
        f"Width={bbox['Width']:.4f}, Height={bbox['Height']:.4f}) | "
        f"Text: {text}"
    )
    print(entry)
    output_lines.append(entry)

# Save to a text file
output_txt_path = "C:\\Users\\RSahu\\Documents\\Azure_Deidentification\\extracted_text_detailed.txt"
with open(output_txt_path, "w", encoding="utf-8") as f:
    f.write("\n".join(output_lines))

print(f"\nSaved detailed output to {output_txt_path}")

In [ ]:
lines

## 6. Word Index (`full_text` + `offset_map`)

`build_word_index()` turns the Textract blocks into the two objects every later section uses:

- **`full_text`**: the page as one string, in reading order (a space between words, a new line after each line). This is the text sent to Azure.
- **`offset_map`**: one entry per word: its text, `start`/`end` position in `full_text`, Textract `word_id`, `line_idx`, confidence, handwriting flag, and
  - `px`: the pixel box (fraction × image size; `floor` for left/top, `ceil` for right/bottom so no letter edge is cut off),
  - `angle`: the tilt in degrees, from the top edge of the word's polygon (converted to pixels first, because a page is not square).

`full_text[start:end]` is always exactly the word, so a position in Azure's text leads straight to a word and its pixels. `word_by_id` is the same list keyed by word id (the next cell only displays it).

In [ ]:
# Word index: rebuild the page text line by line and remember, for every word, where it is
# in the text (start/end) and on the image (pixel box, polygon, tilt), plus its line and type.
import math
from PIL import Image

def build_word_index(blocks, img_w, img_h):
    """Returns (full_text, offset_map). offset_map is in reading order."""
    block_map = {b['Id']: b for b in blocks} 
    line_blocks = [b for b in blocks if b['BlockType'] == 'LINE']

    full_text = ""
    offset_map = []

    for line_idx, line in enumerate(line_blocks):
        word_ids = []
        for rel in line.get('Relationships', []):
            if rel['Type'] == 'CHILD':
                word_ids.extend(rel['Ids'])

        for word_id in word_ids:
            word_block = block_map.get(word_id)
            if not word_block or word_block['BlockType'] != 'WORD':
                continue  # skip SELECTION_ELEMENT or other non-word children

            word_text = word_block['Text']
            bbox = word_block['Geometry']['BoundingBox']
            poly = word_block['Geometry'].get('Polygon') or []

            # Skew of this word in degrees (0 = horizontal), from the top edge of its polygon
            angle = 0.0
            if len(poly) >= 2:
                dx = (poly[1]['X'] - poly[0]['X']) * img_w
                dy = (poly[1]['Y'] - poly[0]['Y']) * img_h
                if abs(dx) + abs(dy) > 0:
                    angle = math.degrees(math.atan2(dy, dx))

            start = len(full_text)
            full_text += word_text
            end = len(full_text)

            offset_map.append({
                "idx": len(offset_map),
                "start": start,
                "end": end,
                "text": word_text,
                "bbox": bbox,
                "confidence": word_block['Confidence'],
                "word_id": word_id,
                "line_idx": line_idx,
                "px": (int(math.floor(bbox['Left'] * img_w)),
                       int(math.floor(bbox['Top'] * img_h)),
                       int(math.ceil((bbox['Left'] + bbox['Width']) * img_w)),
                       int(math.ceil((bbox['Top'] + bbox['Height']) * img_h))),
                "poly": [(pt['X'] * img_w, pt['Y'] * img_h) for pt in poly] if len(poly) >= 3 else None,
                "angle": angle,
                "handwriting": word_block.get('TextType') == 'HANDWRITING',
            })
            full_text += " "   # space between words
        full_text += "\n"      # newline between lines

    return full_text, offset_map


with Image.open(IMAGE_PATH) as _im:
    IMG_W, IMG_H = _im.size

full_text, offset_map = build_word_index(blocks, IMG_W, IMG_H)
word_by_id = {e['word_id']: e for e in offset_map}

print(f"Image: {IMG_W}x{IMG_H} px")
print(f"Total words indexed: {len(offset_map)}")
print(f"Handwritten words: {sum(e['handwriting'] for e in offset_map)}")
print(f"Reconstructed text length: {len(full_text)} characters")
if DEBUG:
    debug_ocr(offset_map)

In [ ]:
word_by_id

## 7. Company Certificates (Netskope)

The company network inspects HTTPS traffic with its own certificate, which Python does not trust by default. This cell builds one certificate file = Python's standard list (`certifi`) + the Netskope certificates from the Windows certificate store, and points `REQUESTS_CA_BUNDLE`, `SSL_CERT_FILE` and `AWS_CA_BUNDLE` at it, so AWS and Azure calls work. Off the company network nothing is added.

In [ ]:
import certifi

def export_netskope_ca():
    """Write a CA bundle containing certifi roots plus any Netskope roots from the Windows store."""
    base_bundle = certifi.where()
    bundle_text = ""
    if base_bundle and os.path.exists(base_bundle):
        with open(base_bundle, "r", encoding="ascii", errors="ignore") as f:
            bundle_text = f.read()
    if sys.platform != "win32":
        return base_bundle if bundle_text else None

    pems = []
    for store in ("ROOT", "CA"):
        for der, encoding, _ in ssl.enum_certificates(store):
            if encoding != "x509_asn":
                continue
            pem = ssl.DER_cert_to_PEM_cert(der)
            if b"netskope" in der.lower() or b"goskope" in der.lower():
                pems.append(pem)

    if not pems:
        return base_bundle if bundle_text else None

    path = os.path.join(tempfile.gettempdir(), "NetskopeRootPlusCertifi.pem")
    with open(path, "w", encoding="ascii") as f:
        if bundle_text:
            f.write(bundle_text.rstrip() + "\n")
        f.write("\n".join(pems))
    return path

def configure_ca_bundle(explicit=None):
    bundle = explicit or os.environ.get("REQUESTS_CA_BUNDLE") or os.environ.get("SSL_CERT_FILE") or os.environ.get("AWS_CA_BUNDLE")
    if not bundle:
        bundle = export_netskope_ca()
        if bundle:
            print(f"Using CA bundle at {bundle}")
    if bundle:
        os.environ["REQUESTS_CA_BUNDLE"] = bundle
        os.environ["SSL_CERT_FILE"] = bundle
        os.environ["AWS_CA_BUNDLE"] = bundle
    else:
        print("No custom CA bundle in use (none provided, none found).")
    return bundle

# Force a freshly generated bundle so the notebook does not keep reusing an older temp path.
ca_bundle = configure_ca_bundle(export_netskope_ca())
print(f"✓ CA bundle configured: {ca_bundle if ca_bundle else 'Default'}")

## 8. Azure Sign-in

Opens a browser window; sign in with an Azure account that has the **DeID Data Owner** role. `credential` keeps the token and renews it automatically, so later calls do not ask again.

In [ ]:
from azure.identity import InteractiveBrowserCredential

print("Opening browser for sign-in...")
print("(A browser window will open. Login with your Azure account that has DeID Data Owner role)\n")

credential = InteractiveBrowserCredential(tenant_id=TENANT_ID, timeout=240)

try:
    token = credential.get_token(SCOPE)
    print("✓ Authentication: OK")
    print(f"Token expires in: {token.expires_on}")
except Exception as exc:
    print(f"❌ FAILED: authentication error:\n{exc}")
    raise

## 9. Azure DeID Call

- `call_azure_deidentification(text, operation)` sends one request to the DeID REST API and returns its JSON answer. v6 uses only `operation = "Surrogate"` (section 10). A token is requested for every call (cached and renewed by `azure-identity`), and HTTP errors print Azure's explanation.
- **Usage counter:** every call adds 1 to `AZURE_USAGE["calls"]` and the UTF-8 size of the text sent to `AZURE_USAGE["input_bytes"]`. Azure bills per MB of text de-identified, so this is the number to price (per page in `batch_summary.csv` as `deid_calls` / `deid_input_bytes`).
- `_split_chunks(text, limit)` cuts text longer than `AZURE_MAX_CHARS` (20,000 characters) at line breaks, so a value is never split between two calls. A normal claim page is one chunk.

In [ ]:
# Azure DeID REST call.
#   * the request field is "operation" ("Surrogate" in v6)
#   * a token is requested for every call (azure-identity caches and renews it), so no 401 after expiry
#   * every call is counted in AZURE_USAGE: Azure bills per MB of text de-identified

AZURE_MAX_CHARS = 20000   # text is sent in chunks of at most this many characters
AZURE_USAGE = {"calls": 0, "input_bytes": 0}   # billed data: counted before every DeID call

def call_azure_deidentification(text, operation="Surrogate"):
    """One Azure DeID request. v6 uses only "Surrogate" (Azure also offers "Tag" and "Redact"). Returns the JSON answer."""
    AZURE_USAGE["calls"] += 1
    AZURE_USAGE["input_bytes"] += len(text.encode("utf-8"))   # size of the text actually sent
    token = credential.get_token(SCOPE)
    body = json.dumps({"inputText": text, "operation": operation}).encode()
    request = urllib.request.Request(
        f"{ENDPOINT}/deid?api-version={API_VERSION}",
        data=body,
        method="POST",
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {token.token}",
        },
    )
    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            return json.loads(response.read())
    except urllib.error.HTTPError as exc:
        print(f"❌ Azure DeID error: {exc.code} {exc.reason}")
        print(exc.read().decode(errors="replace")[:500])
        raise


def _split_chunks(text, limit):
    """Split on line boundaries so no entity is cut in half."""
    chunks, pos = [], 0
    while pos < len(text):
        end = min(len(text), pos + limit)
        if end < len(text):
            nl = text.rfind("\n", pos, end)
            if nl > pos:
                end = nl + 1
        chunks.append((pos, text[pos:end]))
        pos = end
    return chunks

print("✅ Azure DeID function defined (Surrogate is used in section 10)")

## 9b. GPT-5.4 Context (label → value pairs)

Azure misses form values that sit far from their labels in the page text. GPT-5.4 (Azure OpenAI) reads the **page image** and lists every filled-in field as **label + value + category + is_phi**, plus PHI outside labelled fields (`free_text_phi`). GPT never decides positions and never writes fake values; it only gives Azure **context**.

| Step | Function | What happens |
| --- | --- | --- |
| 1 | `vlm_extract` | the image (≤ `VLM_MAX_WIDTH` px wide) + `VLM_PROMPT` → GPT-5.4 → JSON (`fields`, `free_text_phi`, `printed_text`) |
| 2 | `match_text` | each GPT label and value is found among the OCR words (same line, n−1…n+1 words, ≥ 85 % similar after removing case and punctuation; short values must match exactly; date captions `MM DD YY` are skipped while matching) |
| 3 | `vlm_pairs_and_hits` | the value closest to its label is used; each field becomes a pair `{label, value word ids}`; values GPT marked as PHI become **GPT hits** |
| 4 | `build_context_view` | the pairs are written as `label: value` lines (`ctx_text`); only the value words are mapped back to word ids (`ctx_map`) |

Used in section 10: `ctx_text` is sent to Azure Surrogate as a second text, and GPT hits that Azure changed in neither text are erased (`MISSED_PHI_ACTION`).

If the GPT call fails, the page continues with the page text only (`vlm_error` in the summary).

**Saved per page** (`SAVE_GPT_RESULT`, folder `GPT_RESULT_DIR`): `<name>_gpt.json` with GPT's answer exactly as returned, every field with whether and where it was found among the OCR words, the `label: value` text sent to Azure, and the GPT PHI hits (or the error, if the call failed). **It holds real PHI: keep the folder as protected as the original images.**

**Settings:** `SAVE_GPT_RESULT`, `GPT_RESULT_DIR`; (`.env`: `endpoint`, `deployment`, `subscription_key`, `api_version`, optional `model_name`): `USE_VLM`, `VLM_MAX_WIDTH`, `VLM_MAX_TOKENS`, `VLM_MATCH_THRESHOLD`, `VLM_CACHE_DIR` (saves GPT answers for re-runs; the files contain PHI).

In [ ]:
# GPT-5.4 (Azure OpenAI) reads the page image and returns label / value / category for every filled-in field.
# Values are matched back to the Textract words, so the VLM adds CONTEXT, while Textract keeps the POSITIONS.
import base64
import io
import re
from difflib import SequenceMatcher
from pathlib import Path

from openai import AzureOpenAI
from PIL import Image

# ---------------------------------------------------------------- settings
USE_VLM = True
VLM_MAX_WIDTH = 2000          # image is resized to at most this width before sending
VLM_MAX_TOKENS = 16000        # GPT-5.x counts reasoning tokens here too
VLM_MATCH_THRESHOLD = 0.85    # how similar a VLM value must be to OCR words to count as the same text
VLM_CACHE_DIR = None          # e.g. Path(r"C:\...\vlm_cache") to reuse answers on re-runs (contains PHI)

# ---------------------------------------------------------------- Azure OpenAI settings from .env
AZURE_OPENAI_ENDPOINT = os.getenv("endpoint") or os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_DEPLOYMENT = os.getenv("deployment") or os.getenv("AZURE_OPENAI_DEPLOYMENT")
AZURE_OPENAI_API_KEY = os.getenv("subscription_key") or os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("api_version") or os.getenv("AZURE_OPENAI_API_VERSION") or "2024-10-21"
VLM_MODEL_NAME = os.getenv("model_name") or os.getenv("model") or "gpt-5.4"     # for logging only

_missing = [n for n, v in {"endpoint": AZURE_OPENAI_ENDPOINT, "deployment": AZURE_OPENAI_DEPLOYMENT,
                           "subscription_key": AZURE_OPENAI_API_KEY}.items() if not v]
if USE_VLM and _missing:
    raise ValueError(f"Missing Azure OpenAI settings in .env: {', '.join(_missing)} (or set USE_VLM = False)")

vlm_client = AzureOpenAI(api_version=AZURE_OPENAI_API_VERSION, azure_endpoint=AZURE_OPENAI_ENDPOINT,
                         api_key=AZURE_OPENAI_API_KEY) if USE_VLM else None

VLM_PROMPT = """You are reading a scanned healthcare document (for example a CMS-1500 or UB-04 claim form).
Return JSON only, in this shape:
{"fields": [{"label": "...", "value": "...", "category": "...", "is_phi": true}],
 "free_text_phi": [{"value": "...", "category": "..."}],
 "printed_text": ["..."]}

fields: every FILLED-IN field on the page.
- "label": the printed field name exactly as printed, including its box or form-locator number
  (e.g. "2. PATIENT'S NAME (Last Name, First Name, Middle Initial)", "10 BIRTHDATE", "45 SERV. DATE").
- "value": the typed or handwritten value for that field, copied EXACTLY as printed: same spelling, spacing,
  punctuation and date format. Do not correct, reformat or complete anything.
- One entry per field. For tables (e.g. UB-04 service lines) give one entry per filled cell that holds an
  identifier or a date, with the column header as label.
- If one box holds a full address, give street, city, state and zip as separate entries with the same label.
- "category": one of name, date, age_over_89, phone, fax, email, ssn, medical_record_number, health_plan_id,
  account_number, license_number, id_number, street, city, state, zip, country, organization, hospital,
  profession, url, other.
- "is_phi": true for identifiers of a patient, insured person, guarantor, relative or provider (names, dates,
  addresses, phone numbers, ID / account / policy / member / license / NPI numbers); false for diagnosis or
  procedure codes, charges, units, checkbox marks, yes/no, sex, relationship, and printed form text.

free_text_phi: identifiers that are NOT in a labelled field (notes, stamps, signatures, headers), copied exactly.

printed_text: every PRE-PRINTED piece of text that belongs to the blank form itself, one entry per printed line,
copied exactly: form titles, approval lines, box captions and labels, checkbox captions (e.g. MEDICARE, MEDICAID,
TRICARE, SSN, EIN, YES, NO), column headers, date captions (MM DD YY), instructions, footers, form numbers.
Never include anything that was typed or written into the form.

Do not list empty fields. Do not invent values.
Return {"fields": [], "free_text_phi": [], "printed_text": []} if nothing is filled in."""

# VLM category -> kind used by section 12 / 13
VLM_KIND = {
    "name": "name", "date": "date", "phone": "phone", "fax": "phone", "email": "email",
    "street": "street", "city": "city", "state": "state", "zip": "zip", "country": "country",
    "organization": "org", "hospital": "org", "profession": "profession",
}   # everything else (ssn, *_number, health_plan_id, age_over_89, url, other) -> "id"


def vlm_kind(category, value=""):
    c = str(category or "").strip().lower()
    if c == "age_over_89":
        digits = re.sub(r"\D", "", str(value))
        return "id" if digits and int(digits) > 89 else None
    return VLM_KIND.get(c, "id")


# ---------------------------------------------------------------- calling GPT-5.4
def _image_b64(image_path, max_width=VLM_MAX_WIDTH):
    image = Image.open(image_path).convert("RGB")
    if image.width > max_width:
        scale = max_width / image.width
        image = image.resize((int(image.width * scale), int(image.height * scale)), Image.Resampling.LANCZOS)
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")


def _parse_json(text_output):
    cleaned = (text_output or "").strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
        cleaned = re.sub(r"\s*```$", "", cleaned)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", cleaned, re.S)
        if not match:
            raise
        return json.loads(match.group(0))


def vlm_extract(image_path):
    """GPT-5.4 answer for one page: {"fields": [...], "free_text_phi": [...]}. Uses the cache if set."""
    cache = Path(VLM_CACHE_DIR) / f"{Path(image_path).stem}_vlm.json" if VLM_CACHE_DIR else None
    if cache and cache.exists():
        return json.loads(cache.read_text(encoding="utf-8"))
    response = vlm_client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        messages=[
            {"role": "system", "content": "You read healthcare document images and return strict JSON only."},
            {"role": "user", "content": [
                {"type": "text", "text": VLM_PROMPT},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{_image_b64(image_path)}",
                                                    "detail": "high"}},
            ]},
        ],
        max_completion_tokens=VLM_MAX_TOKENS,
    )
    choice = response.choices[0]
    if choice.finish_reason == "length":
        raise RuntimeError(f"VLM output cut off at {VLM_MAX_TOKENS} tokens; raise VLM_MAX_TOKENS")
    result = _parse_json(choice.message.content)
    result.setdefault("fields", []); result.setdefault("free_text_phi", []); result.setdefault("printed_text", [])
    if cache:
        cache.parent.mkdir(parents=True, exist_ok=True)
        cache.write_text(json.dumps(result, indent=1, ensure_ascii=False), encoding="utf-8")
    return result


# ---------------------------------------------------------------- matching VLM text to OCR words
def _compact(s):
    return re.sub(r"[^a-z0-9]", "", str(s or "").lower())

def _similar(a, b):
    if not a or not b:
        return 0.0
    if min(len(a), len(b)) <= 3:
        return 1.0 if a == b else 0.0
    return SequenceMatcher(None, a, b).ratio()

# Date captions printed between date boxes ("MM 05 DD 08 YY 1985") are skipped when matching values.
CAPTION_TOKENS = {"MM", "DD", "YY", "YYYY", "CCYY"}

def _ocr_lines(offset_map):
    by_line = {}
    for w in offset_map:
        if w['text'].upper().strip(".,:;()|[]") in CAPTION_TOKENS:
            continue
        by_line.setdefault(w['line_idx'], []).append(w)
    return [sorted(ws, key=lambda w: w['idx']) for _, ws in sorted(by_line.items())]

def match_text(text_value, offset_map, lines=None):
    """Where `text_value` is printed: [(score, [words])], best first, non-overlapping.
    Windows of n-1..n+1 consecutive words on one OCR line are compared after removing case and punctuation."""
    target = _compact(text_value)
    if not target:
        return []
    n = max(1, len(str(text_value).split()))
    hits = []
    for line in (lines or _ocr_lines(offset_map)):
        for i in range(len(line)):
            for size in range(max(1, n - 1), n + 2):
                win = line[i:i + size]
                if len(win) < size:
                    break
                wc = _compact(" ".join(w['text'] for w in win))
                score = _similar(target, wc)
                if score < VLM_MATCH_THRESHOLD and size == 1 and len(target) >= 4 and target in wc:
                    score = VLM_MATCH_THRESHOLD          # value glued to other text, e.g. "DOB:03/15/1980"
                if score >= VLM_MATCH_THRESHOLD:
                    hits.append((score, -len(win), win))
    chosen, used = [], set()
    for score, _neg, win in sorted(hits, key=lambda h: (-h[0], h[1])):
        ids = {w['word_id'] for w in win}
        if ids & used:
            continue
        used |= ids
        chosen.append((score, win))
    return chosen

def _centre(win):
    return (sum((w['px'][0] + w['px'][2]) / 2 for w in win) / len(win),
            sum((w['px'][1] + w['px'][3]) / 2 for w in win) / len(win))

def _char_spans(win, value):
    """Per word: the characters that hold the value (whole word, or the part inside e.g. 'DOB:03/15/1980')."""
    if len(win) == 1:
        i = win[0]['text'].lower().find(str(value).strip().lower())
        if i >= 0:
            return [(i, i + len(str(value).strip()))]
    return [None] * len(win)


def _printed_word_ids(printed, offset_map, lines):
    """OCR words of the printed lines GPT listed; long lines are also matched in 3-word pieces."""
    ids, matched = set(), 0
    for p in printed:
        p = str(p or "").strip()
        if not _compact(p):
            continue
        found = match_text(p, offset_map, lines)
        words = p.split()
        if not found and len(words) > 3:
            for k in range(0, len(words) - 2):
                found += match_text(" ".join(words[k:k + 3]), offset_map, lines)
        matched += bool(found)
        for _score, win in found:
            ids |= {w['word_id'] for w in win}
    return ids, matched


def vlm_pairs_and_hits(vlm, offset_map):
    """VLM answer -> (pairs in kv_pairs shape, PHI hits for section 12, protected word ids, stats)."""
    lines = _ocr_lines(offset_map)
    pairs, hits, label_ids = [], [], set()
    stats = {"vlm_fields": 0, "vlm_fields_matched": 0, "vlm_free_text": 0, "vlm_free_text_matched": 0,
             "field_rows": [], "free_rows": []}

    for f in vlm.get("fields", []):
        label, value = str(f.get("label", "") or "").strip(), str(f.get("value", "") or "").strip()
        lab_hits = match_text(label, offset_map, lines) if _compact(label) else []
        lab_win = lab_hits[0][1] if lab_hits else []
        lab_ids = {w['word_id'] for w in lab_win}
        label_ids |= lab_ids                     # every label the VLM names is printed form text: protect it
        if not _compact(value):
            continue
        stats["vlm_fields"] += 1
        row = {"label": label, "value": value, "category": f.get("category"), "is_phi": f.get("is_phi", True),
               "matched": "NO"}
        stats["field_rows"].append(row)
        val_hits = [(s, w) for s, w in match_text(value, offset_map, lines) if not ({x['word_id'] for x in w} & lab_ids)]
        if not val_hits:
            continue                                                     # not found among OCR words
        if lab_win:                                                      # the occurrence closest to its label
            lx, ly = _centre(lab_win)
            win = min(val_hits, key=lambda h: (_centre(h[1])[0] - lx) ** 2 + (_centre(h[1])[1] - ly) ** 2)[1]
        else:
            win = val_hits[0][1]
        stats["vlm_fields_matched"] += 1
        row["matched"] = "yes: " + " ".join(w['text'] for w in win)
        pairs.append({"key_text": label or "Value", "key_word_ids": [w['word_id'] for w in lab_win],
                      "value_text": " ".join(w['text'] for w in win), "value_word_ids": [w['word_id'] for w in win]})
        kind = vlm_kind(f.get("category"), value)
        if f.get("is_phi", True) and kind:
            hits.append({"word_ids": [w['word_id'] for w in win], "spans": _char_spans(win, value), "kind": kind})

    for t in vlm.get("free_text_phi", []):
        value = str(t.get("value", "") or "").strip()
        kind = vlm_kind(t.get("category"), value)
        if not _compact(value) or not kind:
            continue
        stats["vlm_free_text"] += 1
        found = match_text(value, offset_map, lines)
        if len(_compact(value)) < 4:
            found = found[:1]                                            # short values: best match only
        for _score, win in found:
            hits.append({"word_ids": [w['word_id'] for w in win], "spans": _char_spans(win, value), "kind": kind})
        stats["vlm_free_text_matched"] += bool(found)
        stats["free_rows"].append({"value": value, "category": t.get("category"), "matched": "yes" if found else "NO"})

    printed_ids, matched = _printed_word_ids(vlm.get("printed_text", []), offset_map, lines)
    label_ids |= printed_ids
    stats["printed_lines"] = len(vlm.get("printed_text", []))
    stats["printed_lines_matched"] = matched
    return pairs, hits, label_ids, stats


def vlm_build_context(image_path, offset_map, word_by_id):
    """Context for section 12 from GPT-5.4 only (no layout pairs mixed in):
    (kv_pairs, label_word_ids, ctx_text, ctx_map, vlm_hits, vlm_label_ids, stats). Raises if the GPT call fails."""
    vlm = vlm_extract(image_path)
    pairs, hits, vlm_label_ids, stats = vlm_pairs_and_hits(vlm, offset_map)
    stats["vlm_raw"] = vlm                                     # GPT's answer as returned, for save_gpt_result
    value_ids = {i for p in pairs for i in p['value_word_ids']} | {i for h in hits for i in h['word_ids']}
    label_ids = ({i for p in pairs for i in p['key_word_ids']} | vlm_label_ids) - value_ids
    ctx_t, ctx_m = build_context_view(pairs, [], word_by_id)
    return pairs, label_ids, ctx_t, ctx_m, hits, vlm_label_ids - value_ids, stats



# Second view of the page for Azure: one "label: value" line per form field and per
# table cell. Dense forms (CMS-1500, UB-04) print the value far from its label, so the
# line-order text alone gives the model little context. Here each value keeps its
# offsets, so whatever Azure tags can be mapped back to the exact word boxes.

def build_context_view(kv_pairs, table_cells, word_by_id):
    ctx_text, ctx_map = "", []

    def add(label, word_ids):
        nonlocal ctx_text
        ids = [i for i in word_ids if i in word_by_id]
        if not ids:
            return
        ctx_text += (label.strip() or "Value") + ": "
        for wid in ids:
            start = len(ctx_text)
            ctx_text += word_by_id[wid]['text']
            ctx_map.append({"start": start, "end": len(ctx_text), "word_id": wid})
            ctx_text += " "
        ctx_text += "\n"

    for p in kv_pairs:
        add(p['key_text'], p['value_word_ids'])
    for c in table_cells:
        add(c['header_text'], c['word_ids'])
    return ctx_text, ctx_map


SAVE_GPT_RESULT = True        # write <name>_gpt.json for every page (GPT's answer + how it was matched)
GPT_RESULT_DIR = Path(r"C:\Users\RSahu\Documents\Azure_deidentification\Input_docs\gpt_results")


def save_gpt_result(vlm_stats, kv_pairs, vlm_hits, ctx_text, word_by_id, stem, out_dir, image_path=""):
    """<stem>_gpt.json: GPT-5.4's answer exactly as returned, every field with whether / where it was found
    among the OCR words, the label: value text sent to Azure, and the GPT PHI hits.
    It holds REAL PHI: keep the folder as protected as the original images."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    record = {
        "image_path": str(image_path),
        "model": VLM_MODEL_NAME, "deployment": AZURE_OPENAI_DEPLOYMENT,
        "error": vlm_stats.get("vlm_error", ""),
        "stats": {k: v for k, v in vlm_stats.items() if not k.endswith("_rows") and k != "vlm_raw"},
        "gpt_answer": vlm_stats.get("vlm_raw"),                        # fields, free_text_phi, printed_text
        "fields_matched": vlm_stats.get("field_rows", []),             # label, value, category, is_phi, matched
        "free_text_matched": vlm_stats.get("free_rows", []),
        "label_value_pairs": [{"label": p["key_text"], "value": p["value_text"],
                               "value_word_ids": p["value_word_ids"]} for p in kv_pairs],
        "label_value_text_sent_to_azure": ctx_text,
        "gpt_phi_hits": [{"kind": h["kind"], "words": [word_by_id[w]['text'] for w in h["word_ids"] if w in word_by_id],
                          "word_ids": h["word_ids"]} for h in vlm_hits],
    }
    path = out_dir / f"{stem}_gpt.json"
    path.write_text(json.dumps(record, indent=1, ensure_ascii=False, default=str), encoding="utf-8")
    return path


def debug_vlm(stats, kv_pairs, vlm_hits):
    """Stage 1b: what GPT-5.4 returned and whether each value was found among the OCR words."""
    _dbg_title("STAGE 1b - GPT-5.4: fields returned, and whether each was found among the OCR words")
    _dbg_rows([(r["label"], r["value"], r["category"], r["is_phi"], r["matched"]) for r in stats.get("field_rows", [])],
              ("label", "value", "category", "is_phi", "matched_to_ocr"))
    print("\nfree-text PHI:")
    _dbg_rows([(r["value"], r["category"], r["matched"]) for r in stats.get("free_rows", [])],
              ("value", "category", "matched_to_ocr"))
    print(f"\nlabel: value lines for Azure: {len(kv_pairs)} | GPT PHI hits: {len(vlm_hits)}")
    print("stats:", {k: v for k, v in stats.items() if not k.endswith("_rows") and k != "vlm_raw"})


# ---------------------------------------------------------------- run for this page
kv_pairs, ctx_text, ctx_map, vlm_hits, vlm_stats = [], "", [], [], {}
if USE_VLM:
    try:
        kv_pairs, _label_ids, ctx_text, ctx_map, vlm_hits, _printed_ids, vlm_stats = vlm_build_context(
            IMAGE_PATH, offset_map, word_by_id)
        print(f"GPT ({VLM_MODEL_NAME}, deployment {AZURE_OPENAI_DEPLOYMENT}): "
              f"{ {k: v for k, v in vlm_stats.items() if not k.endswith('_rows') and k != 'vlm_raw'} }")
        print(f"label: value lines for Azure: {len(kv_pairs)} | GPT PHI hits: {len(vlm_hits)}")
    except Exception as exc:
        vlm_stats = {"vlm_error": repr(exc)[:200]}
        print(f"⚠ GPT step failed, continuing with the page text only: {exc!r}")
else:
    print("USE_VLM = False: Azure gets the page text only.")
if USE_VLM and SAVE_GPT_RESULT:
    _gpt_path = save_gpt_result(vlm_stats, kv_pairs, vlm_hits, ctx_text, word_by_id,
                                os.path.splitext(os.path.basename(IMAGE_PATH))[0], GPT_RESULT_DIR, IMAGE_PATH)
    print(f"Saved: {_gpt_path}")
if DEBUG:
    debug_vlm(vlm_stats, kv_pairs, vlm_hits)

## 10. Fake Values from Azure DeID Surrogate (two texts)

Azure finds the PHI **and** writes the fake values. It now gets **two texts**, and GPT's PHI list is a safety net.

| Step | Function | What happens |
| --- | --- | --- |
| 1 | `_align_view` on **`full_text`** | the page text → Azure Surrogate → lined up with the original **line by line, word by word** (`difflib.SequenceMatcher`); each changed block of OCR words gets Azure's words (`_distribute`) |
| 2 | `_align_view` on **`ctx_text`** | GPT's `label: value` text (section 9b) → Azure Surrogate. Here every value sits next to its label, so Azure recognises values it missed in step 1. Used only for words step 1 left unchanged. Label words have no word id, so Azure can never change a label through this text |
| 3 | GPT safety net | a value GPT marked as PHI of which Azure changed **no word** in either text is erased (`MISSED_PHI_ACTION = "erase"`, only the PHI characters) or kept (`"keep"`), and listed (Stage 2c, `_changes.txt`) |
| 4 | `SURROGATE_MEMO` | the first fake for a value is reused on every later page of the batch, so one person keeps one fake name |
| 5 | `_shape_like` | each new word keeps the original's leading/trailing punctuation and letter case: `SMITH,` → `MONTGOMERY,`, `(717)` → `(512)` |

**All fake values come from Azure.** GPT only adds context (step 2) and points at PHI Azure left (step 3).

**Cost:** two Azure DeID calls per page (both counted in `deid_input_bytes`) + one GPT call (section 9b).

**Text files per page** (`SAVE_SURROGATE_TEXT`, folder `SURROGATE_TEXT_DIR`):
- `<name>_surrogated.txt`: the page text as it appears on the output image.
- `<name>_changes.txt`: every change as `FROM (page / label:value)  ORIGINAL -> FAKE`, plus the GPT PHI Azure did not change. **It holds real values next to their fakes: keep it as protected as the original images.**

**Debug (`DEBUG = True`):** Stage 2t: the page text sent, Azure's output for it, the `label: value` text sent, Azure's output for it, the final text on the image. Stage 2: every change and which text it came from. Stage 2c: GPT PHI Azure did not change. Stage 3: original → fake per value.

**Settings:** `MISSED_PHI_ACTION`, `SAVE_SURROGATE_TEXT`, `SURROGATE_TEXT_DIR`, `REUSE_SURROGATES`, `MATCH_CASE`.

In [ ]:
# Fake values come from Azure DeID's Surrogate operation; this cell maps them onto the OCR words.
# Azure runs on the page text and on GPT's label: value text; every word Azure changed is replaced.
# PHI that GPT found but Azure changed in neither text is erased (MISSED_PHI_ACTION).
import re
from difflib import SequenceMatcher
from pathlib import Path

# ---------------------------------------------------------------- settings
SAVE_SURROGATE_TEXT = True     # write <name>_surrogated.txt and <name>_changes.txt for every page
SURROGATE_TEXT_DIR = Path(r"C:\Users\RSahu\Documents\Azure_deidentification\Input_docs\surrogate_text")
REUSE_SURROGATES = True        # same original value -> same fake on every page of the batch
MATCH_CASE = True              # "SMITH" -> "JOHNSON" (not "Johnson") when the original is upper case


def _compact(s):
    """Lower case, letters and digits only: the key for "same value" in the reuse memory."""
    return re.sub(r"[^a-z0-9]", "", str(s or "").lower())


# ---------------------------------------------------------------- Azure Surrogate
def azure_surrogate_chunks(text):
    """[(base_offset, original_chunk, surrogate_chunk)] from the Surrogate operation."""
    out = []
    for base, chunk in _split_chunks(text, AZURE_MAX_CHARS):
        if not chunk.strip():
            out.append((base, chunk, chunk))
            continue
        result = call_azure_deidentification(chunk, "Surrogate")
        surrogate = result.get("outputText") or result.get("output_text")
        if surrogate is None:
            raise RuntimeError(f"No outputText in the Azure Surrogate response. Keys returned: {list(result)}")
        out.append((base, chunk, surrogate))
    return out


def _distribute(new_tokens, k):
    """Spread Azure's tokens over k original words (extras join the last word, missing ones leave a word empty)."""
    if k <= 0:
        return []
    if len(new_tokens) >= k:
        return new_tokens[:k - 1] + [" ".join(new_tokens[k - 1:])]
    return new_tokens + [""] * (k - len(new_tokens))


_EDGE = r"""[^\w]*"""

def _shape_like(piece, orig):
    """Azure's piece with the original word's leading/trailing punctuation and letter case."""
    if not piece:
        return ""
    lead = re.match(r"^" + _EDGE, orig).group()
    trail = re.search(_EDGE + r"$", orig).group() if orig.strip(".,:;()[]{}'\"-/") else ""
    core = re.sub(r"^" + _EDGE, "", piece)
    core = re.sub(_EDGE + r"$", "", core) or piece
    new = lead + core + trail
    letters = [c for c in orig if c.isalpha()]
    if MATCH_CASE and letters and any(c.isalpha() for c in new):
        if all(c.isupper() for c in letters):
            new = new.upper()
        elif all(c.islower() for c in letters):
            new = new.lower()
    return new


SURROGATE_MEMO = {}      # compact original value -> Azure's words (re-identification key: memory only, never saved)
MISSED_PHI_ACTION = "erase"   # GPT says PHI but Azure changed it in neither text: "erase" (blank it) or "keep"


def _align_view(text, start_to_word, view, rows, counts, parts):
    """Azure Surrogate on one text. Returns {word_id: new text} for every OCR word Azure changed.
    start_to_word maps the position where a VALUE word starts in `text` to its word id; label words of the
    label: value text have no id, so Azure's changes to them are never applied."""
    out = {}
    segments = []
    for base, chunk, surrogate in azure_surrogate_chunks(text):
        parts.append(surrogate)
        c_lines, s_lines = chunk.split("\n"), surrogate.split("\n")
        if len(c_lines) == len(s_lines):              # Azure kept the line breaks: align line by line (safest)
            off = 0
            for cl, sl in zip(c_lines, s_lines):
                segments.append((base + off, cl, sl))
                off += len(cl) + 1
        else:                                          # otherwise align the whole chunk at once
            segments.append((base, chunk, surrogate))

    for seg_base, seg, sur in segments:
        src = [(m.group(), start_to_word.get(seg_base + m.start())) for m in re.finditer(r"\S+", seg)]
        dst = sur.split()
        if [tok for tok, _ in src] == dst:
            continue                                                 # line unchanged
        matcher = SequenceMatcher(None, [tok for tok, _ in src], dst, autojunk=False)
        for op, i1, i2, j1, j2 in matcher.get_opcodes():
            if op == "equal":
                continue
            if op == "insert":                                       # Azure added words: nowhere to draw them
                counts["inserts"] += 1
                continue
            part, new = src[i1:i2], dst[j1:j2]
            if all(wid for _, wid in part):                          # only OCR value words: one block
                groups = [([wid for _, wid in part], [tok for tok, _ in part], new)]
            elif len(part) == len(new):                              # label + value words mixed: value words 1:1
                groups = [([wid], [tok], [n]) for (tok, wid), n in zip(part, new) if wid]
            else:
                if any(wid for _, wid in part):
                    counts["unaligned"] += 1                         # cannot tell which new word belongs where
                continue
            for wids, orig, new_words in groups:
                key = _compact(" ".join(orig))                       # same value seen before -> same fake
                if REUSE_SURROGATES and key:
                    if key in SURROGATE_MEMO and SURROGATE_MEMO[key] != new_words:
                        new_words = SURROGATE_MEMO[key]
                        counts["reused"] += len(wids)
                    else:
                        SURROGATE_MEMO.setdefault(key, new_words)
                for wid, tok, piece in zip(wids, orig, _distribute(new_words, len(wids))):
                    out[wid] = _shape_like(piece, tok)
                rows.append((view, " ".join(orig), " ".join(out[w] for w in wids), wids))
    return out


def apply_azure_surrogates(full_text, offset_map, ctx_text="", ctx_map=None, vlm_hits=None):
    """Sets w['new_text'] on every word to change. Returns (flags, entities, stats).
    1. Azure Surrogate on the page text; 2. Azure Surrogate on the GPT label: value text for words still
    unchanged; 3. GPT PHI that Azure changed in neither text is erased (or kept, MISSED_PHI_ACTION)."""
    word_by_id = {e['word_id']: e for e in offset_map}
    for w in offset_map:
        w.pop("new_text", None)                                    # clean state when a cell is re-run
    rows, counts, page_parts, ctx_parts = [], {"inserts": 0, "unaligned": 0, "reused": 0}, [], []

    # 1. page text
    assigned = _align_view(full_text, {e['start']: e['word_id'] for e in offset_map}, "page", rows, counts, page_parts)
    source = {wid: "azure_page" for wid in assigned}

    # 2. GPT label: value text (values next to their labels) for the words the page text left unchanged
    from_ctx = 0
    if ctx_text.strip() and ctx_map:
        ctx_new = _align_view(ctx_text, {e['start']: e['word_id'] for e in ctx_map}, "label:value", rows, counts, ctx_parts)
        for wid, new_text in ctx_new.items():
            if wid not in assigned and new_text != word_by_id[wid]['text']:
                assigned[wid], source[wid] = new_text, "azure_label_value"
                from_ctx += 1

    # only report changes that were used (label:value changes for words the page text already changed are not)
    rows = [r for r in rows if r[0] == "page" or any(source.get(w) == "azure_label_value" for w in r[3])]
    rows = [r[:3] for r in rows]

    # 3. GPT says PHI, Azure changed it in neither text (if Azure changed part of the value, its decision stands)
    missed = {}
    for h in (vlm_hits or []):
        if any(wid in assigned and assigned[wid] != word_by_id[wid]['text'] for wid in h["word_ids"] if wid in word_by_id):
            continue
        for wid, span in zip(h["word_ids"], h["spans"]):
            w = word_by_id.get(wid)
            if w is None:
                continue
            missed[wid] = (h["kind"], w['text'])
            if MISSED_PHI_ACTION == "erase":
                lo, hi = span if span else (0, len(w['text']))
                assigned[wid], source[wid] = w['text'][:lo] + w['text'][hi:], "erased_gpt_phi"

    # flags + entities in the shape the rendering / verification code expects
    flags = {}
    for wid, new_text in assigned.items():
        w = word_by_id[wid]
        if new_text == w['text']:
            continue
        w["new_text"] = new_text
        flags[wid] = {"kind": "phi", "prio": 2, "conf": 1.0, "mask": [True] * len(w['text']), "sources": {source[wid]}}
    entities = []
    for e in offset_map:
        if e['word_id'] not in flags:
            continue
        last = entities[-1] if entities else None
        if last and last["line_idx"] == e['line_idx'] and last["words"][-1]['idx'] == e['idx'] - 1:
            last["words"].append(e)
        else:
            entities.append({"kind": "phi", "line_idx": e['line_idx'], "words": [e], "conf": 1.0})
    for ent in entities:
        ent["sources"] = sorted(set().union(*(flags[w['word_id']]["sources"] for w in ent["words"])))
        ent["original"] = " ".join(w['text'] for w in ent["words"])
        ent["surrogate"] = " ".join(w['new_text'] for w in ent["words"])

    n_erased = sum(1 for wid in flags if "erased_gpt_phi" in flags[wid]["sources"])
    stats = {"changed_words": len(flags), "changed_values": len(entities),
             "changed_by_page_text": sum(1 for wid in flags if "azure_page" in flags[wid]["sources"]),
             "changed_by_label_value_text": from_ctx,
             "gpt_phi_not_changed_by_azure": len(missed), "erased_words": n_erased,
             "reused_from_memo_words": counts["reused"], "azure_inserts_skipped": counts["inserts"],
             "unaligned_blocks_skipped": counts["unaligned"],
             "rows": rows, "missed": list(missed.values()),
             "surrogate_text": "".join(page_parts), "ctx_text": ctx_text, "ctx_surrogate_text": "".join(ctx_parts)}
    return flags, entities, stats


def final_page_text(offset_map):
    """The page text as drawn on the output image: each word's new_text if it changed, else its OCR text."""
    by_line = {}
    for w in offset_map:
        by_line.setdefault(w['line_idx'], []).append(w.get('new_text', w['text']))
    return "\n".join(" ".join(t for t in ws if t) for _, ws in sorted(by_line.items()))


def save_surrogate_texts(stats, offset_map, stem, out_dir):
    """<stem>_surrogated.txt: the page text as drawn on the output image, one OCR line per line.
    <stem>_changes.txt: original -> fake for every changed value.
    The changes file holds REAL values next to their fakes: keep it as protected as the originals."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    surrogated_path = out_dir / f"{stem}_surrogated.txt"
    surrogated_path.write_text(final_page_text(offset_map) + "\n", encoding="utf-8")

    rows = stats.get("rows", [])
    w = max([len(o) for _, o, _ in rows] + [8])
    lines_out = [f"Changes for {stem}", "=" * 80, "", f"Changed values: {len(rows)}", "",
                 f"{'FROM':<12}  {'ORIGINAL':<{w}}  ->  FAKE (drawn on the image)", "-" * (w + 56)]
    lines_out += [f"{v:<12}  {o:<{w}}  ->  {n}" for v, o, n in rows]
    missed = stats.get("missed", [])
    lines_out += ["", f"GPT says PHI, Azure changed it in neither text ({MISSED_PHI_ACTION}d): {len(missed)}", "-" * 60]
    lines_out += [f"{k}: {txt}" for k, txt in missed]
    changes_path = out_dir / f"{stem}_changes.txt"
    changes_path.write_text("\n".join(lines_out) + "\n", encoding="utf-8")
    return surrogated_path, changes_path


def debug_texts(full_text, stats, offset_map):
    """Stage 2t: what was sent to Azure, what Azure returned, what is drawn on the image."""
    _dbg_title("STAGE 2t-1 - Reconstructed OCR page text (full_text): what is sent to Azure")
    print(full_text)
    _dbg_title("STAGE 2t-2 - Azure Surrogate output (outputText): Azure's text with its fake values")
    print(stats.get("surrogate_text", ""))
    if stats.get("ctx_text", "").strip():
        _dbg_title("STAGE 2t-3 - GPT label: value text (ctx_text): what is sent to Azure as the second text")
        print(stats["ctx_text"])
        _dbg_title("STAGE 2t-4 - Azure Surrogate output for the label: value text")
        print(stats.get("ctx_surrogate_text", ""))
    _dbg_title("STAGE 2t-5 - Final text on the output image")
    print(final_page_text(offset_map))


def debug_alignment(stats):
    """Stage 2: original -> Azure's value for every changed block; 2c: GPT PHI that Azure did not change."""
    _dbg_title("STAGE 2 - Azure Surrogate: original -> Azure's value (per changed block, and which text)")
    _dbg_rows(stats.get("rows", []), ("from_text", "original", "azure_value_drawn"))
    _dbg_title(f"STAGE 2c - GPT says PHI, Azure changed it in neither text ({MISSED_PHI_ACTION})")
    _dbg_rows(stats.get("missed", []), ("gpt_kind", "word"))


flags, entities, surrogate_stats = apply_azure_surrogates(full_text, offset_map, ctx_text, ctx_map, vlm_hits)
print("Azure Surrogate:", {k: v for k, v in surrogate_stats.items()
                           if k not in ("rows", "missed", "surrogate_text", "ctx_text", "ctx_surrogate_text")})
print(f"Azure DeID usage so far: {AZURE_USAGE}")
if SAVE_SURROGATE_TEXT:
    _stem = os.path.splitext(os.path.basename(IMAGE_PATH))[0]
    for _p in save_surrogate_texts(surrogate_stats, offset_map, _stem, SURROGATE_TEXT_DIR):
        print(f"Saved: {_p}")
if DEBUG:
    debug_texts(full_text, surrogate_stats, offset_map)
    debug_alignment(surrogate_stats)
    debug_surrogates(entities)

## 11. Replace on the Image

`render_page()` erases every word Azure changed and draws Azure's value in its place. Principle: **fit on the original, then swap the text.**

| Step | Function | What happens |
| --- | --- | --- |
| Font | `choose_styles`, `score_fit` | the ORIGINAL text is drawn in every candidate font and compared with the real pixels (Dice overlap × shape check); the best font per line wins, near-ties settled by text of the same size elsewhere |
| Size | `fit_size` | the size at which the original text is exactly as wide as its ink; one size per line |
| Handwriting | `style_for_word` | words Textract marks as handwriting use the best-fitting installed handwriting font (`HANDWRITING_FONT_FILES`); a slightly wider area is erased around them |
| Erase | `analyze_word`, `erase_word` | only ink pixels inside the word box are removed (form lines are detected and kept), filled with real paper pixels sampled next to the word |
| Colour | `ink_colour` | the new text carries the same amount of ink as the original (blue pen stays blue) |
| Draw | `word_geometry`, `draw_word` | at the original left edge and baseline, rotated by the word's tilt, softened by a 0.45 px blur |

**Fakes of a different length** (Azure's fakes keep no fixed length), in this order:
1. a longer fake grows into free space to its right, up to `MAX_EXTRA_WIDTH` (100 %) wider, never past the next word or a vertical form line (`_free_right`);
2. if the unchanged words after it are in the way, they are moved along with it, up to a form line (`CARRY_WORDS`);
3. still too long: condensed down to `MIN_SQUEEZE` (75 %), then drawn smaller down to `MIN_FONT_SCALE` (80 %);
4. a shorter fake starts where the original started; the rest of the box stays blank.

The Stage 4 printout and `batch_summary.csv` show `condensed`, `smaller font`, `moved` and `handwriting font` per word. The image is saved as a new PNG built from pixels only (no metadata from the original).

In [ ]:
# Put Azure's fake values on the image. Principle: "fit on the original, then swap the text".
# For every changed word we know both the original text (from OCR) and its pixels, so:
#   1. FONT  - render the ORIGINAL text in each candidate font and score it against
#              the real pixels; keep the best one per line (steadied by other text of
#              the same size), so mixed-font documents work.
#   2. SIZE  - the size at which the original text is exactly as wide as its ink;
#              one size per line, so neighbouring words never differ.
#   3. ERASE - only the ink pixels inside the word box, never the form lines, filled
#              with real background pixels sampled next to the word (works on white,
#              grey and coloured forms).
#   4. DRAW  - each new word at the original word's x position and baseline, in the
#              original ink colour, slightly blurred like a scan.
# Handwritten words use a handwriting font. Fakes longer than the original grow into free
# space, push unchanged words along, then are condensed / drawn smaller (see the section text).
import io
import numpy as np
from collections import Counter as _Counter
from functools import lru_cache
from PIL import Image, ImageDraw, ImageFont, ImageFilter

# ---------------------------------------------------------------- settings
MIN_OCR_CONF = 70.0     # below this the OCR text is not trusted -> solid box
MAX_SKEW_DEG = 3.0      # words rotated more than this -> solid box
ROTATE_MIN_DEG = 0.7    # words rotated more than this are drawn rotated
CALIB_WORDS = 60        # extra non-PHI words sampled to learn the fonts on the page
FONT_PEER_WEIGHT = 0.15 # how strongly same-size text elsewhere settles a near-tie between fonts
BLUR_RADIUS = 0.45      # softening of the drawn text (0 = crisp)
BOX_COLOR = (255, 0, 0)   # colour of the fail-safe box
EXTRA_FONT_PATHS = []   # add full paths of any other .ttf/.otf you want considered
KEEP_LOW_CONFIDENCE = False   # False: low-confidence PHI words are replaced too (True left them visible)

# Handwritten words (Textract TextType = HANDWRITING) are redrawn in a handwriting-style font.
# The best of these fonts is chosen per word by the same fit test as printed text.
# Fonts that are not installed are skipped; with none found, handwriting falls back to the line's printed font.
USE_HANDWRITING_FONT = True
HANDWRITING_FONT_FILES = [
    "Inkfree.ttf", "segoesc.ttf", "BRADHITC.TTF",                          # Windows / Office
    "Caveat-Regular.ttf", "Caveat-VariableFont_wght.ttf",                  # Google Fonts (free)
    "PatrickHand-Regular.ttf", "Kalam-Regular.ttf", "Kalam-Light.ttf",
]
EXTRA_HANDWRITING_FONT_PATHS = []   # full paths of any other handwriting .ttf/.otf
HANDWRITING_ERASE_GROW = 0.2        # erase this fraction of the word height around a handwritten word (loops, tails)

# Fake values that are LONGER than the original (Azure's fakes keep no fixed length), in this order:
#   1. grow into free space to the right (never past the next word or a vertical form line)
#   2. condense horizontally, down to MIN_SQUEEZE
#   3. draw in a smaller font, down to MIN_FONT_SCALE (then condense as much as needed)
# SHORTER fakes start where the original started; the rest of the box stays blank.
MAX_EXTRA_WIDTH = 1.0    # a fake may be up to 100 % wider than the original when the space to its right is free
CARRY_WORDS = True       # a longer fake may push the unchanged words after it (same line) along, up to a form line
MIN_SQUEEZE = 0.75       # condense down to 75 % of the natural width before making the font smaller
MIN_FONT_SCALE = 0.80    # font size may go down to 80 % of the original

FONT_DIRS = [r"C:\Windows\Fonts", os.path.expanduser(r"~\AppData\Local\Microsoft\Windows\Fonts"),
             "/usr/share/fonts", "/Library/Fonts", "/System/Library/Fonts/Supplemental"]
FONT_FILES = [  # typewriter, sans, narrow sans, serif, each regular + bold
    "cour.ttf", "courbd.ttf", "consola.ttf", "consolab.ttf", "lucon.ttf", "OCRAEXT.TTF",
    "arial.ttf", "arialbd.ttf", "arialn.ttf", "arialnb.ttf", "calibri.ttf", "calibrib.ttf",
    "verdana.ttf", "tahoma.ttf", "segoeui.ttf", "times.ttf", "timesbd.ttf",
    # metric-compatible free equivalents (Linux / macOS)
    "LiberationMono-Regular.ttf", "LiberationMono-Bold.ttf", "LiberationSans-Regular.ttf",
    "LiberationSans-Bold.ttf", "LiberationSansNarrow-Regular.ttf", "LiberationSerif-Regular.ttf",
    "LiberationSerif-Bold.ttf", "DejaVuSans.ttf", "DejaVuSansMono.ttf",
]

def find_fonts():
    wanted = {f.lower() for f in FONT_FILES}
    found = {}
    for d in FONT_DIRS:
        if not os.path.isdir(d):
            continue
        for root, _dirs, files in os.walk(d):
            for f in files:
                if f.lower() in wanted and f.lower() not in found:
                    found[f.lower()] = os.path.join(root, f)
    paths = [found[f.lower()] for f in FONT_FILES if f.lower() in found]
    paths += [p for p in EXTRA_FONT_PATHS if os.path.exists(p)]
    if not paths:
        raise RuntimeError("No candidate fonts found. Add font file paths to EXTRA_FONT_PATHS.")
    return paths

FONTS = find_fonts()
print(f"Candidate fonts found: {len(FONTS)} -> {[os.path.basename(p) for p in FONTS]}")

def find_handwriting_fonts():
    wanted = {f.lower() for f in HANDWRITING_FONT_FILES}
    found = {}
    for d in FONT_DIRS:
        if not os.path.isdir(d):
            continue
        for root, _dirs, files in os.walk(d):
            for f in files:
                if f.lower() in wanted and f.lower() not in found:
                    found[f.lower()] = os.path.join(root, f)
    paths = [found[f.lower()] for f in HANDWRITING_FONT_FILES if f.lower() in found]
    paths += [p for p in EXTRA_HANDWRITING_FONT_PATHS if os.path.exists(p)]
    return paths

HANDWRITING_FONTS = find_handwriting_fonts() if USE_HANDWRITING_FONT else []
print(f"Handwriting fonts found: {len(HANDWRITING_FONTS)} -> {[os.path.basename(p) for p in HANDWRITING_FONTS]}"
      + ("" if HANDWRITING_FONTS or not USE_HANDWRITING_FONT else "  (none: handwriting uses the printed font of its line)"))


# ---------------------------------------------------------------- text rendering
@lru_cache(maxsize=4096)
def _font(path, size):
    return ImageFont.truetype(path, size)

@lru_cache(maxsize=100000)
def render_text(text, path, size):
    """Render text; returns (alpha HxW uint8 cropped to the ink, ink_left - origin_x, ink_top - baseline_y)."""
    font = _font(path, size)
    l, t, r, b = font.getbbox(text, anchor="ls")
    l, t, r, b = int(np.floor(l)), int(np.floor(t)), int(np.ceil(r)), int(np.ceil(b))
    pad = 4
    w, h = (r - l) + 2 * pad, (b - t) + 2 * pad
    if w <= 2 * pad or h <= 2 * pad:
        return None
    im = Image.new("L", (w, h), 0)
    ImageDraw.Draw(im).text((pad - l, pad - t), text, font=font, fill=255, anchor="ls")
    a = np.asarray(im)
    ys, xs = np.where(a >= 96)
    if len(xs) == 0:
        return None
    x0, x1, y0, y1 = xs.min(), xs.max() + 1, ys.min(), ys.max() + 1
    return a[y0:y1, x0:x1].copy(), int(x0 - (pad - l)), int(y0 - (pad - t))


# ---------------------------------------------------------------- reading the original pixels
def _otsu(values):
    hist = np.bincount(values.astype(np.uint8).ravel(), minlength=256).astype(float)
    total = hist.sum()
    cum, cum_mean = np.cumsum(hist), np.cumsum(hist * np.arange(256))
    w0, w1 = cum, total - cum
    with np.errstate(divide="ignore", invalid="ignore"):
        m0, m1 = cum_mean / w0, (cum_mean[-1] - cum_mean) / w1
        between = w0 * w1 * (m0 - m1) ** 2
    between = np.where(np.isfinite(between), between, -1.0)
    top = between.max()
    best = np.flatnonzero(between >= top - abs(top) * 1e-6)
    return int((best[0] + best[-1]) // 2)

def _dilate(mask):
    p = np.pad(mask, 1)
    out = np.zeros_like(mask)
    for dy in (0, 1, 2):
        for dx in (0, 1, 2):
            out |= p[dy:dy + mask.shape[0], dx:dx + mask.shape[1]]
    return out

def _runs(column):
    """[(start, end)] of consecutive True values in a 1-D boolean array."""
    padded = np.concatenate([[False], column, [False]])
    edges = np.flatnonzero(padded[1:] != padded[:-1])
    return list(zip(edges[::2], edges[1::2]))

def _find_form_lines(ink):
    """Mask of ruled lines crossing the crop. A form line enters at one edge of the padded
    crop and leaves at the opposite edge (also when the scan is skewed); text never does.
    Text that touches a line is kept: only the thin band of the line itself is marked."""
    h, w = ink.shape
    lines = np.zeros_like(ink)

    def scan(m, out):                      # horizontal-ish lines of m, written into out
        hh, ww = m.shape
        if ww < 12:
            return
        xs = np.arange(ww)
        for s0, e0 in _runs(m[:, 0]):
            for s1, e1 in _runs(m[:, -1]):
                y0, y1 = (s0 + e0 - 1) / 2.0, (s1 + e1 - 1) / 2.0
                if abs(y0 - y1) > max(2.0, 0.06 * ww) or abs((e0 - s0) - (e1 - s1)) > 2:
                    continue
                ys = np.rint(y0 + (y1 - y0) * xs / (ww - 1)).astype(int)
                hit = np.zeros(ww, bool)
                for dy in (-1, 0, 1):
                    hit |= m[np.clip(ys + dy, 0, hh - 1), xs]
                if hit.mean() < 0.92:
                    continue
                half = int(np.ceil(max(e0 - s0, e1 - s1) / 2.0)) + 1
                for dy in range(-half, half + 1):
                    yy = ys + dy
                    ok = (yy >= 0) & (yy < hh)
                    out[yy[ok], xs[ok]] = True

    scan(ink, lines)
    if h >= 8:
        scan(ink.T, lines.T)               # vertical lines
    return lines

def analyze_word(gray, w, pad=2):
    """Find the ink of one word. Only pixels inside the word's (possibly tilted) polygon
    count, and form lines crossing the crop are detected and excluded."""
    x0, y0, x1, y1 = w['px']
    X0, Y0 = max(0, x0 - pad), max(0, y0 - pad)
    X1, Y1 = min(gray.shape[1], x1 + pad), min(gray.shape[0], y1 + pad)
    crop = gray[Y0:Y1, X0:X1]
    if crop.size == 0:
        return None

    inside = np.ones(crop.shape, bool)
    if w.get('poly') and abs(w['angle']) > 0.2:      # tilted word: its box is a rotated rectangle
        cx = sum(p[0] for p in w['poly']) / len(w['poly'])
        cy = sum(p[1] for p in w['poly']) / len(w['poly'])
        pts = [(px + (pad if px >= cx else -pad) - X0, py + (pad if py >= cy else -pad) - Y0) for px, py in w['poly']]
        im = Image.new("L", (crop.shape[1], crop.shape[0]), 0)
        ImageDraw.Draw(im).polygon(pts, fill=255)
        inside = np.asarray(im) > 0

    ink = np.zeros(crop.shape, bool)
    lines = np.zeros(crop.shape, bool)
    if int(crop.max()) - int(crop.min()) >= 40:
        dark = crop <= _otsu(crop)
        lines = _find_form_lines(dark)
        ink = dark & ~lines & inside
    ys, xs = np.where(ink)
    inkbox = (X0 + xs.min(), Y0 + ys.min(), X0 + xs.max() + 1, Y0 + ys.max() + 1) if len(xs) else None
    return {"origin": (X0, Y0), "ink": ink, "lines": lines, "inside": inside, "inkbox": inkbox}

def _tight_mask(info, angle=0.0):
    """The word's ink cropped tight; a skewed word is straightened first so it can be
    compared with horizontally rendered text."""
    X0, Y0 = info["origin"]
    ix0, iy0, ix1, iy1 = info["inkbox"]
    tight = info["ink"][iy0 - Y0:iy1 - Y0, ix0 - X0:ix1 - X0]
    if abs(angle) > ROTATE_MIN_DEG:
        im = Image.fromarray(tight.astype(np.uint8) * 255).rotate(angle, resample=Image.BILINEAR, expand=True)
        a = np.asarray(im) >= 128
        ys, xs = np.where(a)
        if len(xs):
            tight = a[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    return tight

def _ink_size(w, info):
    """(width, height) of the word's ink, measured along the text direction."""
    if not info["inkbox"]:
        return w['px'][2] - w['px'][0], w['px'][3] - w['px'][1]
    tight = _tight_mask(info, w['angle'])
    return tight.shape[1], tight.shape[0]

def _soft(mask_u8):
    return np.asarray(Image.fromarray(mask_u8).filter(ImageFilter.GaussianBlur(0.6)), dtype=float) / 255.0


# ---------------------------------------------------------------- font + size fitting
_REF = 64

def fit_size(text, path, ink_w, ink_h):
    """Font size at which `text` is as wide (or, for very short text, as tall) as its ink."""
    ref = render_text(text, path, _REF)
    if ref is None:
        return None
    rh, rw = ref[0].shape
    by_width = len(text.strip()) >= 3
    guess = _REF * (ink_w / rw if by_width else ink_h / rh)
    guess = int(round(min(300, max(5, guess))))
    best, best_err = None, None
    for s in (guess - 1, guess, guess + 1):
        if s < 5:
            continue
        r = render_text(text, path, s)
        if r is None:
            continue
        err = abs(r[0].shape[1] - ink_w) if by_width else abs(r[0].shape[0] - ink_h)
        if best_err is None or err < best_err:
            best, best_err = s, err
    return best

def score_fit(text, path, size, tight):
    """0..1: how well `text` in this font/size reproduces the word's real ink."""
    r = render_text(text, path, size)
    if r is None:
        return 0.0
    H, W = tight.shape
    h, w = r[0].shape
    rendered = _soft(np.asarray(Image.fromarray(r[0]).resize((W, H), Image.BILINEAR)))
    target = _soft(tight.astype(np.uint8) * 255)
    dice = 2 * np.minimum(rendered, target).sum() / (rendered.sum() + target.sum() + 1e-6)
    # proportions must agree too (1 px of tolerance, since small text is quantised)
    shape_penalty = max(0.0, 1 - 2 * max(0, abs(h - H) - 1) / max(H, 1)) * \
                    max(0.0, 1 - 2 * max(0, abs(w - W) - 1) / max(W, 1))
    return float(dice * shape_penalty)

def _fittable(w):
    return (not w['handwriting'] and abs(w['angle']) <= MAX_SKEW_DEG and w['confidence'] >= 90
            and sum(ch.isalnum() for ch in w['text']) >= 3)

def choose_styles(gray, offset_map, phi_word_ids):
    """Returns (main_font, {line_idx: style}). Fits fonts on the ORIGINAL text of sample words.

    A line's font is the one that best reproduces its own words. Near-ties are settled by
    which font wins most often among other text of a similar size, so a line holding only
    a number does not flip between, say, regular and bold from one box to the next.
    """
    phi_lines = {w['line_idx'] for w in offset_map if w['word_id'] in phi_word_ids}
    in_lines = [w for w in offset_map if w['line_idx'] in phi_lines and _fittable(w)]
    others = [w for w in offset_map if w['line_idx'] not in phi_lines and _fittable(w)]
    step = max(1, len(others) // CALIB_WORDS)
    calib = in_lines + others[::step][:CALIB_WORDS]

    fits = {}          # word_id -> {font: (score, size)}
    size_class = {}    # word_id -> bucket of its best-fit size
    for w in calib:
        info = analyze_word(gray, w)
        if not info or not info["inkbox"]:
            continue
        tight = _tight_mask(info, w['angle'])
        per_font = {}
        for path in FONTS:
            size = fit_size(w['text'], path, tight.shape[1], tight.shape[0])
            if size:
                per_font[path] = (score_fit(w['text'], path, size, tight), size)
        if per_font:
            fits[w['word_id']] = per_font
            best = max(per_font, key=lambda f: per_font[f][0])
            size_class[w['word_id']] = int(round(np.log(per_font[best][1]) / np.log(1.15)))

    def mean_scores(word_ids):
        out = {}
        for path in FONTS:
            vals = [fits[i][path][0] for i in word_ids if path in fits[i]]
            if vals:
                out[path] = float(np.mean(vals))
        return out

    def make_style(members, font):
        sizes = [fits[w['word_id']][font][1] for w in members if font in fits[w['word_id']]]
        return {"font": font, "size": int(round(float(np.median(sizes)))),
                "cx": float(np.mean([(w['px'][0] + w['px'][2]) / 2 for w in members])),
                "cy": float(np.mean([(w['px'][1] + w['px'][3]) / 2 for w in members]))}

    by_line = {}
    for w in calib:
        if w['word_id'] in fits:
            by_line.setdefault(w['line_idx'], []).append(w)

    # Pass 1: lines with enough letters to tell fonts apart.
    line_styles, weak = {}, []
    for line_idx, members in by_line.items():
        ids = [w['word_id'] for w in members]
        own = mean_scores(ids)
        if sum(ch.isalpha() for w in members for ch in w['text']) < 5:
            weak.append((line_idx, members, own))
            continue
        cls = int(np.median([size_class[i] for i in ids]))
        peers = [i for i in fits if abs(size_class[i] - cls) <= 1]
        votes = _Counter(max(fits[i], key=lambda f: fits[i][f][0]) for i in peers)
        font = max(own, key=lambda f: own[f] + FONT_PEER_WEIGHT * votes.get(f, 0) / len(peers))
        line_styles[line_idx] = make_style(members, font)

    # Pass 2: lines that are mostly digits ("62704", "03/15/2024"). Digits look alike in many
    # fonts, so take the font of the nearest lettered line of the same size, provided it
    # also reproduces this line about as well as this line's own best guess.
    strong = dict(line_styles)
    for line_idx, members, own in weak:
        best = max(own, key=own.get)
        style = make_style(members, best)
        near = sorted(strong.values(), key=lambda st: abs(st["cy"] - style["cy"]) * 3 + abs(st["cx"] - style["cx"]))
        for st in near:
            if st["font"] in own and own[st["font"]] >= own[best] - 0.08:
                candidate = make_style(members, st["font"])
                if abs(candidate["size"] - st["size"]) <= 0.15 * st["size"]:
                    style = candidate
                    break
        line_styles[line_idx] = style

    used = [st["font"] for i, st in line_styles.items() if i in phi_lines] or [st["font"] for st in line_styles.values()]
    main_font = _Counter(used).most_common(1)[0][0] if used else FONTS[0]
    return main_font, line_styles

def style_for_word(w, info, doc_font, line_styles):
    """Font and size for one PHI word: its own line's style, else the nearest line whose
    style reproduces this word's ink height (short values like "04" cannot be fitted alone).
    Handwritten words: the handwriting font that best reproduces the original strokes."""
    if w['handwriting'] and HANDWRITING_FONTS and info and info["inkbox"]:
        tight = _tight_mask(info, w['angle'])
        best = None
        for path in HANDWRITING_FONTS:
            size = fit_size(w['text'], path, tight.shape[1], tight.shape[0])   # size from the real ink
            if size:
                score = score_fit(w['text'], path, size, tight)                # same Dice score as printed text
                if best is None or score > best[0]:
                    best = (score, path, size)
        if best:
            return best[1], best[2]
    x0, y0, x1, y1 = w['px']
    ink_w, ink_h = _ink_size(w, info)
    own = line_styles.get(w['line_idx'])
    if own:
        font, size = own["font"], own["size"]
        ref = render_text(w['text'], font, size)
        if ref is not None and len(w['text'].strip()) >= 3 and abs(ref[0].shape[1] - ink_w) > 0.25 * ink_w:
            size = fit_size(w['text'], font, ink_w, ink_h) or size      # this word is a different size
        return font, size
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    best, best_d = None, None
    for st in line_styles.values():
        ref = render_text(w['text'], st["font"], st["size"])
        if ref is None or abs(ref[0].shape[0] - ink_h) > 0.15 * ink_h:
            continue
        d = abs(st["cy"] - cy) * 3 + abs(st["cx"] - cx)                 # prefer lines just above/below
        if best_d is None or d < best_d:
            best, best_d = st, d
    if best:
        return best["font"], best["size"]
    return doc_font, fit_size(w['text'], doc_font, ink_w, ink_h) or max(6, int(0.8 * (y1 - y0)))


# ---------------------------------------------------------------- erase + draw
def erase_word(out, orig, info, box, rng, page_bg):
    """Remove the word's ink (not the form lines) and fill with nearby background pixels."""
    X0, Y0 = info["origin"]
    h, w = info["ink"].shape
    x0, y0, x1, y1 = box
    mask = _dilate(info["ink"]) 
    if not info['ink'].any():
        mask=np.ones_like(mask)                     # also take the grey anti-aliased edge
    allowed = np.zeros_like(mask)                    # stay inside the word box (+1 px; more for handwriting)
    g = max(1, int(round(HANDWRITING_ERASE_GROW * (y1 - y0)))) if info.get("handwriting") else 1
    allowed[max(0, y0 - g - Y0):max(0, y1 + g - Y0), max(0, x0 - g - X0):max(0, x1 + g - X0)] = True
    mask &= allowed & ~info["lines"] & info["inside"]
    if not mask.any():
        return
    background = ~_dilate(_dilate(info["ink"])) & ~_dilate(info["lines"])
    region_orig = orig[Y0:Y0 + h, X0:X0 + w]
    samples = region_orig[background]
    if len(samples) < 8:
        samples = page_bg
    out[Y0:Y0 + h, X0:X0 + w][mask] = samples[rng.integers(0, len(samples), int(mask.sum()))]

def ink_colour(orig, info, ref):
    """Colour that makes the redrawn text carry the same amount of ink as the original.
    (A plain 'darkest pixel' is too dark, a median too light: thin scanned strokes are grey.)"""
    X0, Y0 = info["origin"]
    h, w = info["ink"].shape
    region = orig[Y0:Y0 + h, X0:X0 + w].astype(float)
    near = _dilate(info["ink"])
    background = ~_dilate(near) & ~_dilate(info["lines"])
    if not info["ink"].any() or background.sum() < 8:
        return np.array([0.0, 0.0, 0.0]), None
    bg = np.median(region[background], axis=0)
    near &= ~info["lines"]
    if ref is None:
        return np.median(region[info["ink"]], axis=0), bg
    mass = np.clip(bg - region[near], 0, None).sum(axis=0)          # total darkness of the original word
    coverage = ref[0].astype(float).sum() / 255.0                   # total coverage of the same word redrawn
    return np.clip(bg - mass / max(coverage, 1.0), 0, 255), bg

def word_geometry(w, info, font_path, size):
    """Where the original text sits: (ink_left, ink_right, baseline_y) in image pixels."""
    x0, y0, x1, y1 = w['px']
    ref = render_text(w['text'], font_path, size)
    if info["inkbox"] and ref is not None:
        ix0, iy0, ix1, iy1 = info["inkbox"]
        r_alpha, _r_left, r_top = ref
        baseline = int(round(((iy0 - r_top) + (iy1 - (r_top + r_alpha.shape[0]))) / 2.0))
        return ix0, ix1, baseline
    return x0, x1, y1 - int(round(0.2 * (y1 - y0)))

def draw_word(out, w, font_path, size, colour, left, baseline, squeeze=1.0):
    """Draw w['new_text'] with its ink starting at x=`left` on `baseline`."""
    new = render_text(w['new_text'], font_path, size)
    if new is None:
        return "replaced"
    patch, _n_left, n_top = new
    top = baseline + n_top
    note = "replaced"
    if squeeze < 0.995:                              # squeeze horizontally rather than shrink the font
        width = max(2, int(round(patch.shape[1] * squeeze)))
        patch = np.asarray(Image.fromarray(patch).resize((width, patch.shape[0]), Image.BILINEAR))
        note = "replaced (condensed)"

    im = Image.fromarray(np.pad(patch, 3))
    if abs(w['angle']) > ROTATE_MIN_DEG:
        centre_y = top + patch.shape[0] / 2.0
        im = im.rotate(-w['angle'], resample=Image.BICUBIC, expand=True)
        top = int(round(centre_y - im.size[1] / 2.0)) + 3
    if BLUR_RADIUS > 0:
        im = im.filter(ImageFilter.GaussianBlur(BLUR_RADIUS))
    alpha = np.asarray(im, dtype=float) / 255.0
    left, top = int(round(left)) - 3, top - 3

    H, W = out.shape[:2]
    ax0, ay0 = max(0, -left), max(0, -top)
    ax1, ay1 = min(alpha.shape[1], W - left), min(alpha.shape[0], H - top)
    if ax1 <= ax0 or ay1 <= ay0:
        return "outside image"
    a = alpha[ay0:ay1, ax0:ax1, None]
    region = out[top + ay0:top + ay1, left + ax0:left + ax1].astype(float)
    out[top + ay0:top + ay1, left + ax0:left + ax1] = np.clip(region * (1 - a) + colour * a, 0, 255).astype(np.uint8)
    return note


def _free_right(gray, x_from, y0, y1, x_to):
    """x of the first vertical form line between x_from and x_to on rows y0..y1 (x_to when there is none)."""
    x_from, x_to = max(0, int(x_from)), min(gray.shape[1], int(x_to))
    band = gray[max(0, y0):max(0, y1), x_from:x_to]
    if band.size == 0 or band.shape[1] < 2 or int(band.max()) - int(band.min()) < 40:
        return x_to
    dark = band <= _otsu(band)
    cols = np.flatnonzero(dark.mean(axis=0) >= 0.9)          # a column that is dark over the whole line height
    return x_from + int(cols[0]) if len(cols) else x_to


# ---------------------------------------------------------------- page level
def render_page(image_path, offset_map, flags):
    """Returns (PIL image, actions, doc_font). actions = {word_id: "replaced" | "boxed: reason" | "unchanged"}."""
    source = Image.open(image_path)
    keep_grey = source.mode in ("1", "L")           # save a grey scan as grey, not as a larger RGB file
    img = source.convert("RGB")
    orig = np.array(img)
    out = orig.copy()
    gray = np.asarray(img.convert("L"))
    rng = np.random.default_rng(0)
    flat = orig.reshape(-1, 3)
    bright = flat[gray.ravel() >= np.percentile(gray, 60)]
    page_bg = bright[:: max(1, len(bright) // 5000)]

    phi = [w for w in offset_map if w['word_id'] in flags]
    doc_font, line_styles = choose_styles(gray, offset_map, set(flags))

    # next word to the right on the same line, to know how much room there is
    next_x = {}
    by_line = {}
    for w in offset_map:
        by_line.setdefault(w['line_idx'], []).append(w)
    for ws in by_line.values():
        ws = sorted(ws, key=lambda w: w['px'][0])
        for a, b in zip(ws, ws[1:]):
            next_x[a['word_id']] = b['px'][0]

    actions, boxes, draws = {}, [], []
    for w in phi:
        if w.get('new_text', w['text']) == w['text']:
            actions[w['word_id']] = "unchanged"
            continue
        reason = None
        if w['handwriting']:
            reason = "handwriting"
        elif w['confidence'] < MIN_OCR_CONF:
            reason = "low OCR confidence"
        elif abs(w['angle']) > MAX_SKEW_DEG:
            reason = "rotated text"
        info = analyze_word(gray, w)
        erase_info = info
        if w['handwriting']:                           # erase a little wider for loops and tails (measuring stays tight)
            pad = max(2, int(round(HANDWRITING_ERASE_GROW * (w['px'][3] - w['px'][1]))))
            erase_info = analyze_word(gray, w, pad=pad) or info
            if erase_info is not None:
                erase_info["handwriting"] = True
        if reason == "low OCR confidence" and KEEP_LOW_CONFIDENCE:
            actions[w['word_id']] = "kept: low OCR confidence"
            #boxes.append(w)
            continue
        font_path, size = style_for_word(w, info, doc_font, line_styles)
        ink_left, ink_right, baseline = word_geometry(w, info, font_path, size)
        colour, _bg = ink_colour(orig, info, render_text(w['text'], font_path, size))
        new = render_text(w['new_text'], font_path, size)
        draws.append({"w": w, "info": info, "erase_info": erase_info, "font": font_path, "size": size, "colour": colour,
                      "ink_left": ink_left, "ink_right": ink_right, "baseline": baseline,
                      "new_w": new[0].shape[1] if new is not None else 0})

    def make_draw(w):
        """Measurements for redrawing an UNCHANGED word somewhere else on its line (carried along by a longer fake)."""
        info = analyze_word(gray, w)
        if info is None or not info["inkbox"]:
            return None
        font_path, size = style_for_word(w, info, doc_font, line_styles)
        ink_left, ink_right, baseline = word_geometry(w, info, font_path, size)
        colour, _bg = ink_colour(orig, info, render_text(w['text'], font_path, size))
        r = render_text(w['text'], font_path, size)
        w["new_text"] = w['text']                              # drawn as it was, only moved
        return {"w": w, "info": info, "erase_info": info, "font": font_path, "size": size, "colour": colour,
                "ink_left": ink_left, "ink_right": ink_right, "baseline": baseline,
                "new_w": r[0].shape[1] if r is not None else 0, "carried": True}

    line_words = {li: sorted(ws, key=lambda w: w['px'][0]) for li, ws in by_line.items()}

    # Lay out runs of consecutive replaced words: the first word starts where the original
    # started and the ORIGINAL gaps between words are kept, so typewriter text lands exactly
    # on the old positions and proportional text keeps natural spacing. If a run would reach
    # the next untouched word it is squeezed horizontally; if it is shorter, the spare room
    # is shared between its gaps.
    runs = []
    for d in draws:
        prev = runs[-1][-1] if runs else None
        if prev and prev["w"]['line_idx'] == d["w"]['line_idx'] and prev["w"]['idx'] == d["w"]['idx'] - 1 \
                and d["ink_left"] >= prev["ink_right"]:
            runs[-1].append(d)
        else:
            runs.append([d])
    page_w = gray.shape[1]

    def _layout(run):
        """Place the run's words at the original gaps; returns the right end of the last word."""
        x = run[0]["ink_left"]
        for k, d in enumerate(run):
            if k:
                x += run[k - 1]["new_w"] + (d["ink_left"] - run[k - 1]["ink_right"])
            d["left"] = x
        return run[-1]["left"] + run[-1]["new_w"]

    def _room(run, start):
        """Right limit for a run: MAX_EXTRA_WIDTH wider at most, never past the next word or a form line."""
        orig_right = run[-1]["ink_right"]
        last = run[-1]["w"]
        height = last['px'][3] - last['px'][1]
        gap = max(2, int(0.25 * height))
        next_left = next_x.get(last['word_id'], page_w)
        wall = _free_right(gray, orig_right + 1, last['px'][1], last['px'][3], next_left)
        limit = min(orig_right + MAX_EXTRA_WIDTH * (orig_right - start) + 0.5 * height, wall - gap, next_left - gap)
        return max(limit, orig_right), orig_right

    extra = []
    for run in runs:
        start, end = run[0]["ink_left"], _layout(run)
        limit, orig_right = _room(run, start)
        if CARRY_WORDS and end > limit:
            # the next words on the line are unchanged: move them along with the fake, up to a form line
            last = run[-1]["w"]
            ws = line_words.get(last['line_idx'], [])
            pos = next((k for k, x in enumerate(ws) if x['word_id'] == last['word_id']), None)
            height = last['px'][3] - last['px'][1]
            wall = _free_right(gray, orig_right + 1, last['px'][1], last['px'][3], page_w)
            carried = []
            for x in (ws[pos + 1:] if pos is not None else []):
                if x['word_id'] in flags or x['px'][2] > wall:     # stop at the next replaced word or a form line
                    break
                d = make_draw(x)
                if d is None:
                    break
                carried.append(d)
            if carried:
                run.extend(carried)
                extra.extend(carried)
                end = _layout(run)
                limit, orig_right = _room(run, start)
        ratio = (limit - start) / max(end - start, 1)
        scale = 1.0
        if ratio < MIN_SQUEEZE:                                   # too long even when condensed: smaller font
            scale = max(MIN_FONT_SCALE, ratio / MIN_SQUEEZE)
            for d in run:
                d["size"] = max(5, int(round(d["size"] * scale)))
                r = render_text(d["w"]['new_text'], d["font"], d["size"])
                d["new_w"] = r[0].shape[1] if r is not None else 0
            end = _layout(run)
            ratio = (limit - start) / max(end - start, 1)
        squeeze = min(1.0, ratio)
        slack = max(0.0, orig_right - end) if len(run) > 1 else 0.0   # shorter multi-word run: share the room between gaps
        for k, d in enumerate(run):
            d["left"] = start + (d["left"] - start) * squeeze + slack * k / len(run)
            d["squeeze"] = squeeze
            d["smaller"] = scale < 0.995
    draws += extra                                         # carried words are erased and redrawn too
    
    # erase everything first, then draw, so one word's eraser never touches another's new text
    for w in boxes:
        x0, y0, x1, y1 = w['px']
        out[max(0, y0):y1, max(0, x0):x1] = BOX_COLOR
    for d in draws:
        erase_word(out, orig, d["erase_info"], d["w"]['px'], rng, page_bg)
    for d in draws:
        note = draw_word(out, d["w"], d["font"], d["size"], d["colour"], d["left"], d["baseline"], d["squeeze"])
        if d.get("carried"):
            note = note.replace("replaced", "moved (unchanged text)", 1)
        if d.get("smaller"):
            note += " (smaller font)"
        if d["font"] in HANDWRITING_FONTS:
            note += " (handwriting font: " + os.path.basename(d["font"]) + ")"
        actions[d["w"]['word_id']] = note

    result = Image.fromarray(out)
    return (result.convert("L") if keep_grey else result), actions, doc_font


deid_image, actions, doc_font = render_page(IMAGE_PATH, offset_map, flags)   # only words Azure changed

OUTPUT_IMAGES_DIR = os.path.join(os.path.dirname(IMAGE_PATH), "output_images")
os.makedirs(OUTPUT_IMAGES_DIR, exist_ok=True)
_stem = os.path.splitext(os.path.basename(IMAGE_PATH))[0]
deid_image_path = os.path.join(OUTPUT_IMAGES_DIR, f"{_stem}_deid.png")
deid_image.save(deid_image_path)      # new pixels only: no metadata from the original is carried over

_summary = _Counter(a.split(":")[0].split(" (")[0] for a in actions.values())
print(f"Main font used for the replacements: {os.path.basename(doc_font)}")
print(f"Words: {dict(_summary)}")
print(f"Saved: {deid_image_path}")
if DEBUG:
    debug_render(actions, word_by_id)

## 11b. Side-by-side Check (before the batch)

Shows the test image `IMAGE_PATH` next to its de-identified version (original left, de-identified right) and saves it as `<name>_comparison.png` in `output_images`. Check that every PHI value is replaced and that the fake values sit in the right place, in a matching font, before running the batch.

In [ ]:
# Original | de-identified, for the single test image (sections 4-11), before running the batch.
import matplotlib.pyplot as plt

def side_by_side(original_path, deid_img, label_band=56, gap=24):
    """One image: original on the left, de-identified on the right, each with a label above it."""
    original = Image.open(original_path).convert("RGB")
    deid_img = deid_img.convert("RGB")
    width, height = max(original.width, deid_img.width), max(original.height, deid_img.height)
    canvas = Image.new("RGB", (width * 2 + gap, height + label_band), "white")
    canvas.paste(original, (0, label_band))
    canvas.paste(deid_img, (width + gap, label_band))
    draw = ImageDraw.Draw(canvas)
    try:
        label_font = ImageFont.truetype(FONTS[0], 32)
    except Exception:
        label_font = ImageFont.load_default()
    draw.text((12, 10), "Original", fill="black", font=label_font)
    draw.text((width + gap + 12, 10), "De-identified (Azure Surrogate)", fill="black", font=label_font)
    return canvas

comparison = side_by_side(IMAGE_PATH, deid_image)
comparison_path = os.path.join(OUTPUT_IMAGES_DIR, f"{_stem}_comparison.png")
comparison.save(comparison_path)
print(f"Saved: {comparison_path}")
print(f"Changed values: {surrogate_stats['changed_values']} | words: {dict(_summary)}")

plt.figure(figsize=(20, 12))
plt.imshow(comparison)
plt.axis("off")
plt.show()

## 12. Batch: Settings and Functions

| Setting / function | Purpose |
| --- | --- |
| `BATCH_INPUT_DIR`, `BATCH_RECURSIVE` | folder with the input images (and its sub-folders) |
| `BATCH_COMPARISON_DIR` | where comparison images and `batch_summary.csv` go |
| `SAVE_VERIFICATION_RECORD`, `VERIFY_DIR` | per page, a JSON with every original word, what was changed and a re-OCR of the output, used by the leak-verification notebook. **It holds real values next to their fakes: keep it as protected as the originals.** Costs one extra Textract call per page |
| `SAVE_SURROGATE_TEXT`, `SURROGATE_TEXT_DIR` (section 10) | per page, `<name>_surrogated.txt` (Azure's text) and `<name>_changes.txt` (original -> fake). **The changes file holds real values: keep it protected** |
| `SAVE_GPT_RESULT`, `GPT_RESULT_DIR` (section 9b) | per page, `<name>_gpt.json`: GPT's answer and how each field was matched. **Holds real PHI: keep it protected** |
| `discover_images()` | finds the images, skipping the notebook's own outputs (`output_images`, `_deid`, `_comparison`) |
| `save_comparison_image()` | original and de-identified image side by side |
| `save_verification_record()` | writes the verification JSON |
| `process_image_pipeline()` | sections 4–11 for one page: OCR → word index → GPT-5.4 context → Azure Surrogate (two texts) → redraw → save; returns one summary row |

**Summary columns:** `changed_words` / `changed_values`, `replaced_words`, `condensed_words`, `smaller_font_words`, `unchanged_words`, `changed_by_page_text` / `changed_by_label_value_text` (which Azure text changed the word), `gpt_phi_not_changed_by_azure` / `erased_words` (GPT safety net), `vlm_used`, `vlm_fields` / `vlm_fields_matched`, `vlm_phi_hits`, `vlm_error`, `unaligned_blocks_skipped`, `reused_from_memo_words`, `azure_inserts_skipped`, `handwritten_phi_words` / `handwriting_font_words`, `font_used`, `verification_record`, `deid_calls` / `deid_input_bytes` (for the DeID cost).

In [ ]:
from pathlib import Path
import csv

# Batch configuration: process every supported image under the input folder and save
# both the de-identified image and a side-by-side comparison image.
BATCH_INPUT_DIR = Path(r"C:\Users\RSahu\Documents\Azure_deidentification\Input_docs\data")
BATCH_COMPARISON_DIR = Path(r"C:\Users\RSahu\Documents\Azure_deidentification\Input_docs\compared_data")
BATCH_RECURSIVE = True

# Verification record per page, used by the phi_leak_verification notebook to find missed PHI.
# It holds the ORIGINAL PHI text next to its fake value, i.e. it is a re-identification key:
# keep this folder as protected as the original images and never share it with the outputs.
SAVE_VERIFICATION_RECORD = True
VERIFY_DIR = Path(r"C:\Users\RSahu\Documents\Azure_deidentification\Input_docs\verification_records")

def discover_images(root_dir, recursive=True):
    root_dir = Path(root_dir)
    pattern = "**/*" if recursive else "*"
    images = []
    for path in root_dir.glob(pattern):
        if not path.is_file():
            continue
        if path.parent.name.lower() == "output_images":
            continue
        if path.stem.lower().endswith("_deid") or path.stem.lower().endswith("_comparison"):
            continue
        if path.suffix.lower() in SUPPORTED_IMAGE_FORMATS:
            images.append(path)
    return sorted(images)

def save_comparison_image(original_path, deid_path, comparison_path):
    original = Image.open(original_path).convert("RGB")
    deidentified = Image.open(deid_path).convert("RGB")

    width = max(original.width, deidentified.width)
    height = max(original.height, deidentified.height)
    label_band = 48
    gap = 24

    canvas = Image.new("RGB", (width * 2 + gap, height + label_band), "white")
    canvas.paste(original, (0, label_band))
    canvas.paste(deidentified, (width + gap, label_band))

    draw = ImageDraw.Draw(canvas)
    label_font = ImageFont.load_default()
    draw.text((12, 14), "Original", fill="black", font=label_font)
    draw.text((width + gap + 12, 14), "De-identified", fill="black", font=label_font)

    comparison_path.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(comparison_path)
    return comparison_path

def _ocr_bytes_for(image):
    """Bytes for Textract (< 10 MB). A PNG re-save of a JPG scan can be larger than the
    original, so fall back to a high-quality JPEG for the OCR call only."""
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    if buf.tell() < 10 * 1024 * 1024:
        return buf.getvalue()
    buf = io.BytesIO()
    image.convert("RGB").save(buf, format="JPEG", quality=95)
    return buf.getvalue()

def save_verification_record(image_path, deid_image, deid_image_path, offset_map, flags, entities, actions, record_path):
    """Re-OCR the de-identified image and save, for every word, the original text, whether it
    was flagged, its fake text and what happened on the image, plus the words read back from
    the output. Nothing is judged here; the verification notebook does that."""
    deid_blocks = get_ocr(_ocr_bytes_for(deid_image))["Blocks"]
    _deid_text, deid_words = build_word_index(deid_blocks, deid_image.width, deid_image.height)

    word_entity = {}
    for n, ent in enumerate(entities):
        for w in ent["words"]:
            word_entity[w['word_id']] = n

    record = {
        "image_path": str(image_path),
        "deid_image_path": str(deid_image_path),
        "image_size": [deid_image.width, deid_image.height],
        "original_words": [{
            "word_id": w['word_id'], "idx": w['idx'], "line_idx": w['line_idx'], "text": w['text'],
            "px": list(w['px']), "confidence": round(w['confidence'], 2), "handwriting": w['handwriting'],
            "flagged": w['word_id'] in flags,
            "kind": flags[w['word_id']]["kind"] if w['word_id'] in flags else None,
            "sources": sorted(flags[w['word_id']]["sources"]) if w['word_id'] in flags else [],
            "new_text": w.get('new_text') if w['word_id'] in flags else None,
            "action": actions.get(w['word_id']) if w['word_id'] in flags else None,
            "entity": word_entity.get(w['word_id']),
        } for w in offset_map],
        "entities": [{
            "n": n, "kind": ent["kind"], "original": ent["original"], "surrogate": ent.get("surrogate"),
            "sources": ent["sources"], "word_ids": [w['word_id'] for w in ent["words"]],
        } for n, ent in enumerate(entities)],
        "deid_words": [{
            "idx": w['idx'], "line_idx": w['line_idx'], "text": w['text'], "px": list(w['px']),
            "confidence": round(w['confidence'], 2),
        } for w in deid_words],
    }
    record_path.parent.mkdir(parents=True, exist_ok=True)
    record_path.write_text(json.dumps(record, indent=1), encoding="utf-8")
    return record_path

def process_image_pipeline(image_path, batch_input_dir, comparison_root):
    AZURE_USAGE.update(calls=0, input_bytes=0)          # count this page only
    image_path = Path(load_image(str(image_path)))

    with open(image_path, "rb") as file_handle:
        image_bytes = file_handle.read()
    assert len(image_bytes) < 10 * 1024 * 1024, f"Image exceeds 10MB sync limit: {image_path.name}"

    response = get_ocr(image_bytes)                                        # Textract basic OCR
    blocks = response["Blocks"]
    with Image.open(image_path) as source_image:
        img_w, img_h = source_image.size
    full_text, offset_map = build_word_index(blocks, img_w, img_h)
    word_by_id = {entry["word_id"]: entry for entry in offset_map}

    kv_pairs, ctx_text, ctx_map, vlm_hits, vlm_stats = [], "", [], [], {}
    if USE_VLM:                                                            # GPT-5.4 context (section 9b)
        try:
            kv_pairs, _label_ids, ctx_text, ctx_map, vlm_hits, _printed_ids, vlm_stats = vlm_build_context(
                image_path, offset_map, word_by_id)
        except Exception as exc:
            print(f"    ⚠ GPT step failed for {image_path.name}, continuing with the page text only: {exc!r}")
            vlm_stats = {"vlm_error": repr(exc)[:200]}

    rel_dir = image_path.parent.relative_to(batch_input_dir) if image_path.parent != batch_input_dir else Path()
    gpt_path = ""
    if USE_VLM and SAVE_GPT_RESULT:
        gpt_path = save_gpt_result(vlm_stats, kv_pairs, vlm_hits, ctx_text, word_by_id, image_path.stem,
                                   GPT_RESULT_DIR / rel_dir, image_path)

    flags, entities, surrogate_stats = apply_azure_surrogates(full_text, offset_map, ctx_text, ctx_map, vlm_hits)
    if DEBUG_BATCH:
        print(f"\n##### {image_path.name}")
        debug_ocr(offset_map)
        debug_vlm(vlm_stats, kv_pairs, vlm_hits)
        debug_texts(full_text, surrogate_stats, offset_map)
        debug_alignment(surrogate_stats)
        debug_surrogates(entities)

    surrogated_path = changes_path = ""
    if SAVE_SURROGATE_TEXT:
        rel = image_path.parent.relative_to(batch_input_dir) if image_path.parent != batch_input_dir else Path()
        surrogated_path, changes_path = save_surrogate_texts(surrogate_stats, offset_map, image_path.stem,
                                                            SURROGATE_TEXT_DIR / rel)

    deid_image, actions, doc_font = render_page(str(image_path), offset_map, flags)
    if DEBUG_BATCH:
        debug_render(actions, word_by_id)

    output_images_dir = image_path.parent / "output_images"
    output_images_dir.mkdir(parents=True, exist_ok=True)
    deid_image_path = output_images_dir / f"{image_path.stem}_deid.png"
    deid_image.save(deid_image_path)

    relative_dir = image_path.parent.relative_to(batch_input_dir) if image_path.parent != batch_input_dir else Path()
    comparison_path = comparison_root / relative_dir / f"{image_path.stem}_comparison.png"
    save_comparison_image(image_path, deid_image_path, comparison_path)

    action_counts = Counter(action.split(":")[0].split(" (")[0] for action in actions.values())
    deid_calls, deid_bytes = AZURE_USAGE["calls"], AZURE_USAGE["input_bytes"]   # before the verification re-OCR

    verify_path = ""
    if SAVE_VERIFICATION_RECORD:
        verify_path = VERIFY_DIR / relative_dir / f"{image_path.stem}_verify.json"
        save_verification_record(image_path, deid_image, deid_image_path, offset_map, flags,
                                 entities, actions, verify_path)

    return {
        "image_path": str(image_path),
        "deid_image_path": str(deid_image_path),
        "comparison_path": str(comparison_path),
        "ocr_words": len(offset_map),
        "changed_words": len(flags),
        "changed_values": len(entities),
        "replaced_words": action_counts.get("replaced", 0),
        "condensed_words": sum(1 for a in actions.values() if "condensed" in a),
        "smaller_font_words": sum(1 for a in actions.values() if "smaller font" in a),
        "unchanged_words": action_counts.get("unchanged", 0),
        "changed_by_page_text": surrogate_stats["changed_by_page_text"],
        "changed_by_label_value_text": surrogate_stats["changed_by_label_value_text"],
        "gpt_phi_not_changed_by_azure": surrogate_stats["gpt_phi_not_changed_by_azure"],
        "erased_words": surrogate_stats["erased_words"],
        "unaligned_blocks_skipped": surrogate_stats["unaligned_blocks_skipped"],
        "vlm_used": bool(USE_VLM and "vlm_error" not in vlm_stats),
        "vlm_fields": vlm_stats.get("vlm_fields", 0),
        "vlm_fields_matched": vlm_stats.get("vlm_fields_matched", 0),
        "vlm_phi_hits": len(vlm_hits),
        "vlm_error": vlm_stats.get("vlm_error", ""),
        "reused_from_memo_words": surrogate_stats["reused_from_memo_words"],
        "azure_inserts_skipped": surrogate_stats["azure_inserts_skipped"],
        "handwritten_phi_words": sum(1 for w in offset_map if w['word_id'] in flags and w['handwriting']),
        "handwriting_font_words": sum(1 for a in actions.values() if "handwriting font" in a),
        "font_used": os.path.basename(doc_font),
        "verification_record": str(verify_path),
        "surrogated_text": str(surrogated_path),
        "changes_text": str(changes_path),
        "gpt_result": str(gpt_path),
        "deid_calls": deid_calls,
        "deid_input_bytes": deid_bytes,
    }

print(f"Batch input folder: {BATCH_INPUT_DIR}")
print(f"Comparison output folder: {BATCH_COMPARISON_DIR}")
print(f"Images discovered right now: {len(discover_images(BATCH_INPUT_DIR, recursive=BATCH_RECURSIVE))}")

## 13. Batch Run

Runs `process_image_pipeline()` on every image found. A page that fails is logged in `batch_errors.csv` and the loop continues. At the end it writes `batch_summary.csv` (one row per page) and prints the total Azure DeID usage (calls, bytes, MB, average per page). The "same value → same fake" memory is reset at the start of each batch.

Outputs per page: `<name>_deid.png` (in `output_images` next to the input), `<name>_comparison.png` (in `BATCH_COMPARISON_DIR`), `<name>_verify.json` (in `VERIFY_DIR`), `<name>_surrogated.txt` and `<name>_changes.txt` (in `SURROGATE_TEXT_DIR`), `<name>_gpt.json` (in `GPT_RESULT_DIR`), all with the same sub-folders as the input.

In [ ]:
batch_images = discover_images(BATCH_INPUT_DIR, recursive=BATCH_RECURSIVE)
print(f"Found {len(batch_images)} image(s) under {BATCH_INPUT_DIR}")

batch_results = []
batch_errors = []
SURROGATE_MEMO.clear()        # fresh "same value -> same fake" memory for this batch

for index, image_path in enumerate(batch_images, start=1):
    print(f"[{index}/{len(batch_images)}] Processing {image_path.name}")
    try:
        result = process_image_pipeline(image_path, BATCH_INPUT_DIR, BATCH_COMPARISON_DIR)
        batch_results.append(result)
        print(
            f"    saved deid={result['deid_image_path']} | comparison={result['comparison_path']} | "
            f"phi_entities={result['phi_entities']} | replaced_words={result['replaced_words']}"
        )
    except Exception as exc:
        batch_errors.append({"image_path": str(image_path), "error": repr(exc)})
        print(f"    FAILED: {exc!r}")

BATCH_COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
summary_csv_path = BATCH_COMPARISON_DIR / "batch_summary.csv"
error_csv_path = BATCH_COMPARISON_DIR / "batch_errors.csv"

if batch_results:
    with open(summary_csv_path, "w", newline="", encoding="utf-8") as file_handle:
        writer = csv.DictWriter(file_handle, fieldnames=list(batch_results[0].keys()))
        writer.writeheader()
        writer.writerows(batch_results)

if batch_errors:
    with open(error_csv_path, "w", newline="", encoding="utf-8") as file_handle:
        writer = csv.DictWriter(file_handle, fieldnames=["image_path", "error"])
        writer.writeheader()
        writer.writerows(batch_errors)

print()
print(f"Completed: {len(batch_results)} image(s)")
print(f"Failed: {len(batch_errors)} image(s)")
print(f"Summary CSV: {summary_csv_path}")
total_bytes = sum(r["deid_input_bytes"] for r in batch_results)
print(f"Azure DeID: {sum(r['deid_calls'] for r in batch_results)} calls, "
      f"{total_bytes:,} bytes = {total_bytes / 1_000_000:.3f} MB "
      f"(avg {total_bytes / max(len(batch_results), 1) / 1000:.1f} KB per page)")
if batch_errors:
    print(f"Error CSV:   {error_csv_path}")

batch_results[:3]

## 14. Preview

Shows one comparison image (`PREVIEW_INDEX`) from the batch results.

In [ ]:
# Preview one saved comparison image after the batch run.
import matplotlib.pyplot as plt
PREVIEW_INDEX = 0

if not batch_results:
    raise RuntimeError("No batch results available. Run the batch cell above first.")

preview_result = batch_results[PREVIEW_INDEX]
preview_path = preview_result["comparison_path"]
print(f"Previewing {preview_path}")

preview_img = Image.open(preview_path).convert("RGB")
plt.figure(figsize=(18, 10))
plt.imshow(preview_img)
plt.axis("off")
plt.show()